# Import library and support function

In [212]:
# #!/usr/bin/env python3

# import os
# import sys
# import gpustat
# import random

# stats = gpustat.GPUStatCollection.new_query()
# ids = map(lambda gpu: int(gpu.entry['index']), stats)
# ratios = map(lambda gpu: float(gpu.entry['memory.used'])/float(gpu.entry['memory.total']), stats)
# pairs = list(zip(ids, ratios))
# random.shuffle(pairs)
# bestGPU = min(pairs, key=lambda x: x[1])[0]

# print(f'Setting GPU to {bestGPU}', file=sys.stderr)
# os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
# os.environ['CUDA_VISIBLE_DEVICES'] = str(bestGPU)
# print(f"export CUDA_VISIBLE_DEVICES={bestGPU}")

In [213]:
import sys
sys.path.insert(0, "/home/jupyter-hanx/SVM_review/utils")
sys.path.insert(0, "/home/jupyter-hanx/SVM_review/")
from utils.DataLoader.dataset_BoT_IoT import BoT_IoT
from utils.DataLoader.dataset_CIC_DDoS_2019 import CIC_DDoS_2019
from utils.DataLoader.dataset_CIC_IDS_2017 import CICIDS2017
from utils.load_data_ids import load_data_ids

from sklearn import svm, base, ensemble, metrics, model_selection, preprocessing, tree, linear_model, datasets
from sklearn.preprocessing import KernelCenterer, StandardScaler, LabelEncoder, OneHotEncoder, QuantileTransformer, MinMaxScaler, Normalizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (ConfusionMatrixDisplay, roc_auc_score, precision_score, average_precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error, roc_curve, auc, classification_report,auc,confusion_matrix,matthews_corrcoef)
from sklearn.datasets import make_blobs, make_multilabel_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier, LocalOutlierFactor, NearestCentroid, RadiusNeighborsClassifier
from sklearn.mixture import BayesianGaussianMixture
from sklearn.svm import SVC, NuSVC
from sklearn.naive_bayes import GaussianNB

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.utils import resample
from sklearn.impute import SimpleImputer


from lightgbm import LGBMClassifier
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM

import plotly.express as px

import os
import numpy as np
import scipy as sp
import pandas as pd
import urllib.request
import shutil
import tarfile
import seaborn as sns
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

from scipy.sparse.linalg import cg

# import tensorflow as tf
from tqdm.notebook import trange, tqdm

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve, cg, bicgstab, gmres, lsqr

import matplotlib.pyplot as plt
import seaborn as sns
import time
import logging
from typing import List, Tuple, Generator, Iterator
import random
from decimal import *

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 



## Support function

In [214]:
#@title Hàm đo chỉ số
# basic random seed
__CUSTOM_COLS=['MCC', 'ACC','TPR', 'FPR', 'F1', 'TPR macro','PPV macro','F1 macro',"AUC","Training time","Testing time"]
__DEFAULT_RANDOM_SEED = 42

def seedEverything(seed=__DEFAULT_RANDOM_SEED):
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)
  np.random.seed(seed)

  # # tensorflow random seed 
  # import tensorflow as tf 
  # tf.random.set_seed(seed)
    
  # # torch random seed
  # import torch
  # torch.manual_seed(seed)
  # torch.cuda.manual_seed(seed)
  # torch.backends.cudnn.deterministic = True
  # torch.backends.cudnn.benchmark = False



def cfs_matrix(y_label, y_pred, labels):
  
    cfs_mt = np.full((len(labels), len(labels)), 0)

    for x,y in zip(y_label, y_pred):
        cfs_mt[x,y] += 1
    
    return cfs_mt

def calc_index(testdf, Label_name, export_fig):

  #     P
  #     0   1
  # T 0 TN FP
  #   1 FN TP


    cnf_matrix = confusion_matrix(testdf.label, testdf.y_pred, labels = np.unique(testdf.label))
  #   # plot_confusion_matrix("Model_cfs",cnf_matrix, target_names=Label_name,figsize = (20, 10), export_fig=export_fig)

    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix) 
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)

    FPR = FP/(FP+TN) *100.

    ACC = (TP+TN)/(TP+FP+FN+TN) *100.

    auc_func = -1
    if len(Label_name) == 2:
        false_positive_rate, true_positive_rate, thresholds = roc_curve(testdf.label,  testdf.y_pred)
        auc_func = auc(false_positive_rate, true_positive_rate)

    tpr_func = recall_score(testdf.label,testdf.y_pred,average='macro', zero_division = 0) *100. 
    ppv_func = precision_score(testdf.label,testdf.y_pred,average='macro', zero_division = 0) *100.
    f1_func = f1_score(testdf.label,testdf.y_pred,average='macro', zero_division = 0) *100.
    mcc_func = matthews_corrcoef(testdf.label,testdf.y_pred)

    ACC = sum(TP+TN)/(sum(TP+FP+FN+TN)) *100.

    FPR = sum(FP)/sum((FP+TN)) *100.

    
    # print("TPR-macro: {:.4f}".format(tpr_func))
    # print("FPR      : {:.4f}".format(FPR))
    # print("PPV-macro: {:.4f}".format(ppv_func))
    # print("F1-macro : {:.4f}".format(f1_func))
    # print("MCC-func  : {:.4f}".format(mcc_func))
    # print("AUC-func  : {:.4f}".format(auc_func))
    # print("CFS MATRIX:\n",cnf_matrix)
    
    return mcc_func, ACC, tpr_func, FPR, ppv_func, f1_func, auc_func, cnf_matrix    

def Average(lst):
  return sum(lst) / len(lst)


def plot_confusion_matrix(name,cm,
                          target_names,
                          title='Confusion matrix',
                          figsize=(15,8),
                          cmap=None,
                          normalize=False,export_fig=None):

  import matplotlib.pyplot as plt
  import numpy as np
  import itertools

  accuracy = np.trace(cm) / np.sum(cm).astype('float')
  misclass = 1 - accuracy

  if cmap is None:
    cmap = plt.get_cmap('Blues')
  norm_cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]*100
  plt.figure(figsize=figsize)
  plt.imshow(norm_cm, interpolation='nearest', cmap=cmap)
  plt.colorbar()

  if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=45, fontsize='large')
    plt.yticks(tick_marks, target_names, fontsize='large')

  thresh = cm.max() / 1.5 if normalize else cm.max() / 2
  thresh= 50
  for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):  
    plt.text(j, i, "{:,}\n{:0.2f}%".format(cm[i, j], norm_cm[i, j]),
                horizontalalignment="center",
                verticalalignment="center",
                color="white" if norm_cm[i, j] > thresh else "black")

  plt.tight_layout()
  plt.ylabel('True label', fontsize='x-large')
  plt.xlabel('Predicted label', fontsize='x-large')

  if export_fig is not None:
    plt.savefig(os.path.join(export_fig,'cfs_matrix.png'), bbox_inches='tight')
  plt.show()


def Export_report(id_information=[], clf_report=None, cnf_matrix=None, custom_report=None, Label_name=[], path="", mode='w'):
  
  df_header = pd.DataFrame([["BEGIN TEST"],[""],id_information,[""]])
  df_row = pd.DataFrame([[""]])
  df_end = pd.DataFrame(["","END TEST",""])
  df_1 = df_2 = df_3 = pd.DataFrame()

  if clf_report is not None:
    df_tmp = pd.DataFrame(clf_report).transpose()
    df_tmp = pd.concat([df_tmp.columns.to_frame().T, df_tmp], ignore_index=False)
    df_tmp.reset_index(inplace=True)
    df_tmp = df_tmp.rename(columns = {'index':''})
    df_1 = pd.concat([df_tmp, df_row], ignore_index=True)
    df_1.set_axis([*range(df_1.shape[1])],axis = 1, inplace=True)
  
  if cnf_matrix is not None:
    df_2 = pd.DataFrame(cnf_matrix)
    df_2.insert(0,'',Label_name)
    df_tmp = pd.DataFrame([Label_name])
    df_2 = pd.concat([df_tmp,df_2,df_row], axis = 0, ignore_index=True)
    df_2.set_axis([*range(df_2.shape[1])],axis = 1, inplace=True)
  if custom_report is not None:
    df_3 = pd.concat([custom_report.columns.to_frame().T, custom_report], ignore_index=False)
    df_3.reset_index(inplace=True)
    df_3 = df_3.rename(columns = {'index':'Model'})
    df_3.set_axis([*range(df_3.shape[1])],axis = 1, inplace=True)

  
  pd.concat([df_header, df_1, df_2, df_3, df_end], ignore_index=True).to_csv(path, header=False, index=False, mode=mode)

TupleOrList = tuple([Tuple, List])
class CustomMerger(base.BaseEstimator, base.TransformerMixin):
  """Merge List of DataFrames"""

  def __init__(self):
    pass

  def fit(self, X: pd.DataFrame, y=None):
    return self

  def transform(self, X: pd.DataFrame, y='deprecated', copy=True):
    if isinstance(X, TupleOrList):
        return pd.concat(X, ignore_index=True).reset_index(drop=True)

    return X


def plot_2d_features(X_train, y_train):
    # Apply PCA for dimensionality reduction to 2 features
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_train)

    # Create a scatter plot
    unique_labels = np.unique(y_train)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_labels)))

    plt.figure(figsize=(8, 6))

    for label, color in zip(unique_labels, colors):
        plt.scatter(X_2d[y_train == label, 0], X_2d[y_train == label, 1], color=color, label=str(label))

    plt.title('2D Feature Plot')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend()
    plt.show()


def plot_tsne_2d(X_train, y_train, perplexity = 30):
    if X_train.shape[1] > 50:
        X_train, pca = apply_pca(X_train)
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    X_tsne = tsne.fit_transform(X_train)
    
    # Create a scatter plot
    # unique_labels = np.unique(y_train)
    # labels = [str(x) for x in unique_labels]
    fig = px.scatter(x=X_tsne[:, 0], y=X_tsne[:, 1],color=y_train)
    
    # fig.update_traces(marker_size=8)
    fig.update_layout(height=900)
    fig.update_layout(width=900)
    fig.show(renderer='iframe')


    # fig = px.scatter(data_frame  =X_tsne, x=0, y=1, color=y_train, labels=labels)
    # fig.show()

def plot_tsne_3d(X_train, y_train, perplexity = 30):
    # Apply t-SNE for dimensionality reduction to 2 features

    if X_train.shape[1] > 50:
        X_train, pca = apply_pca(X_train)
        
    tsne = TSNE(n_components=3, random_state=42, perplexity=perplexity)
    X_tsne = tsne.fit_transform(X_train)
    
    # Create a scatter plot

    fig = px.scatter_3d(x=X_tsne[:, 0], y=X_tsne[:, 1], z=X_tsne[:, 2],color=y_train)
    
    # fig.update_traces(marker_size=8)
    fig.update_layout(height=900)
    fig.update_layout(width=900)
    fig.show(renderer='iframe')


def remove_outliers_lof(X_data, y_data, contamination=0.05, random_seed=None):
    """
    Remove outliers from a dataset using Local Outlier Factor (LOF).

    Parameters:
    - X_data: numpy array, feature matrix
    - y_data: numpy array, label array
    - contamination: float, the proportion of outliers in the dataset
    - random_seed: int or None, seed for reproducibility

    Returns:
    - X_no_outliers: numpy array, feature matrix without outliers
    - y_no_outliers: numpy array, label array without outliers
    """

    unique_classes = np.unique(y_data)

    X_no_outliers = np.empty((0, X_data.shape[1]), dtype=X_data.dtype)
    y_no_outliers = np.empty(0, dtype=y_data.dtype)

    for label in unique_classes:
        # Select samples belonging to the current class
        # print(label)
        class_mask = (y_data == label)
        X_class = X_data[class_mask]
        if label == 0:
            X_no_outliers = np.vstack((X_no_outliers, X_class))
            y_no_outliers = np.concatenate((y_no_outliers, y_data[class_mask]))
        else:
            # Apply LOF to detect outliers
            lof = LocalOutlierFactor(contamination=contamination)
            outliers_mask = lof.fit_predict(X_class) == -1

            # Remove outliers from the current class
            X_no_outliers = np.vstack((X_no_outliers, X_class[~outliers_mask]))
            y_no_outliers = np.concatenate((y_no_outliers, y_data[class_mask][~outliers_mask]))

    return X_no_outliers, y_no_outliers


def calculate_mean_point(cluster_data):
    """
    Calculate the mean point of a cluster.

    Parameters:
    - cluster_data: A list of vectors (NumPy arrays) representing the data points in the cluster.

    Returns:
    - mean_point: The mean point of the cluster.
    """

    # Convert the list of vectors to a NumPy array for efficient computation
    cluster_array = np.array(cluster_data)

    # Calculate the mean along each dimension (axis=0)
    mean_point = np.mean(cluster_array, axis=0)

    return mean_point



def apply_pca(input_array):
    """
    Perform PCA on a NumPy array and reduce its dimensionality.

    Parameters:
    - input_array: The input NumPy array.
    - n_components: The number of components (dimensions) to reduce to.

    Returns:
    - reduced_array: The NumPy array with reduced dimensionality.
    """
    # if n_components >= input_array.shape[1]:
    #     raise ValueError("Number of components should be less than the input array's number of features.")

    # Create PCA instance
    pca = PCA()

    # Fit and transform the input array
    reduced_array = pca.fit_transform(input_array)
    return reduced_array, pca


def apply_lda(X,y):
    """
    Perform PCA on a NumPy array and reduce its dimensionality.

    Parameters:
    - input_array: The input NumPy array.
    - n_components: The number of components (dimensions) to reduce to.

    Returns:
    - reduced_array: The NumPy array with reduced dimensionality.
    """
    # if n_components > min(X.shape[1], len(np.unique(y)) -1):
    #     raise ValueError("Number of components should be less than the input array's number of features.")

    # Create LDA instance
    lda = LinearDiscriminantAnalysis()

    # Fit and transform the input array
    reduced_array = lda.fit_transform(X,y)

    return reduced_array, lda

def visualize_multiclass_dataset(df):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x="Fts_0", y="Fts_1", hue="Label", data=df, palette="Set1", s=50, edgecolor='w')
    plt.title("Multi-Class Dataset with Anomalies")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend(title="Label", loc='upper right')
    plt.show()





## Function to Load Dataset

In [215]:
__DEVICES = ['Danmini_Doorbell', 'Ecobee_Thermostat', 'Ennio_Doorbell', 'Philips_B120N10_Baby_Monitor', 'Provision_PT_737E_Security_Camera', 'Provision_PT_838_Security_Camera', 'Samsung_SNH_1011_N_Webcam', 'SimpleHome_XCS7_1002_WHT_Security_Camera', 'SimpleHome_XCS7_1003_WHT_Security_Camera']


def Import_dataset(data_dir, devices = __DEVICES, bot_type = 2, atk_type = "ALL"):
    
    bot_names = ['mirai','gafgyt']
    __atk_names = {'mirai': ['ack','scan','syn','udp','udpplain'],
                 'gafgyt': ['combo','junk','scan','tcp','udp']
                }
    atk_names = []
    
    if bot_type == 0:
      bot_names = ['mirai']
    elif bot_type == 1:
      bot_names = ['gafgyt']
    
    if atk_type == "ALL":
      for x in bot_names:
        atk_names = atk_names + __atk_names[x]
    else:
      atk_names = atk_type
    
    
    print("Load normal file:")
    
    df_nors = []
    for device_name in devices:
      device_id = __DEVICES.index(device_name) + 1
      file_path = os.path.join(data_dir,'{}.benign.csv'.format(device_id))
      print("Load file benign:", file_path)
      if os.path.exists(file_path) == False:
        print('File {} not found. Ignored'.format(file_path))
        continue
        
      normal = pd.read_csv(file_path)
      # dropping duplicate values 
      normal.drop_duplicates(keep=False, inplace=True) 

      normal['Names Atk'] = np.array(['Benign']*normal.shape[0])
      normal['Names Bot'] = np.array(['Benign']*normal.shape[0])
      normal['Devices'] = np.array([device_name]*normal.shape[0])
      df_nors.append(normal)
      
    # CustomMerger.transform(df_nors)
    df_nors = pd.concat(df_nors, ignore_index=True).reset_index(drop=True)
    df_nors['Label'] = np.array([0]*df_nors.shape[0])
    
    print("Normal shape:", df_nors.shape)
    
    print("Load abnormal file:")
    
    df_anos = []
    for device_name in devices:
      device_id = __DEVICES.index(device_name) + 1
      df_bots = []
      for bot in bot_names:
        df_atks = []
        for atk in __atk_names[bot]:
          if atk not in atk_names:
            continue
          file_path = os.path.join(data_dir,'{}.{}.{}.csv'.format(device_id,bot,atk))
          print("Load file benign:", file_path)
          if os.path.exists(file_path) == False:
            print("File {} not found. Ignore".format(file_path))
            continue
          atk_sample = pd.read_csv(file_path)
          atk_sample.drop_duplicates(keep=False, inplace=True) 
            
          atk_sample['Names Atk'] = np.array([atk]*atk_sample.shape[0])
          df_atks.append(atk_sample) 
                                   
        # CustomMerger.transform(df_atks)
        if df_atks:
          df_atks = pd.concat(df_atks, ignore_index=True).reset_index(drop=True)
          df_atks['Names Bot'] = np.array([bot]*df_atks.shape[0])
          df_bots.append(df_atks)
                                   
      # CustomMerger.transform(df_bots)
      if df_bots:
        df_bots = pd.concat(df_bots, ignore_index=True).reset_index(drop=True)
        df_bots['Devices'] = np.array([device_name]*df_bots.shape[0])
        df_anos.append(df_bots)  
                                   
    # CustomMerger.transform(df_anos)
    if df_anos:
      df_anos = pd.concat(df_anos, ignore_index=True).reset_index(drop=True)
      df_anos['Label'] = np.array([1]*df_anos.shape[0])
    print("Abnormal shape:", df_anos.shape)
    
    return df_nors, df_anos

def Preprocess_data(df):
  df.drop(columns=['Label','Names Bot','Devices'], inplace=True, errors='ignore')
  df.rename(columns={'Names Atk': __TARGET}, inplace=True)
  return df

def Get_sample_ratio_by_column(df, col, max_cnt):
  dfse = df[col].value_counts()
  ans = []
  for x in dfse.index:
    tmp = df[df[col] == x].sample(n = max_cnt, random_state=__SEED)
    ans.append(tmp)
  return pd.concat(ans, ignore_index=True).reset_index(drop=True) 



def Train_test_split_unsup(df, rate):
    global __UNSUP, __LIST_ATK
    __UNSUP = "UNSUP"
    __LIST_ATK = df[__TARGET].unique()
    
    __LIST_ATK = __LIST_ATK[ (__LIST_ATK != "BENIGN") & (__LIST_ATK != "Normal") ]

    df_normal = df[df[__TARGET].isin(['BENIGN', 'Normal'])]
    df_attack = df[df[__TARGET].isin(__LIST_ATK)]
    
    df_normal[__TARGET] = df_normal[__TARGET].apply(lambda x: 0 if (x == 'BENIGN' or x == 'Normal') else 1)
    df_attack[__TARGET] = df_attack[__TARGET].apply(lambda x: 0 if (x == 'BENIGN' or x == 'Normal') else 1)
    
    df_train_nor, df_test_nor = train_test_split(df_normal, test_size=rate, random_state=__SEED)
    
    df_test = pd.concat([df_test_nor, df_attack], axis=0)
    
    
    return df_train_nor, df_test

def Train_test_split_one_atk_unsup(df, rate):
    global __UNSUP, __LIST_ATK
    __UNSUP = "UNSUP"
    __LIST_ATK = df[__TARGET].unique()
    
    __LIST_ATK = __LIST_ATK[ (__LIST_ATK != "BENIGN") & (__LIST_ATK != "Normal") ]

    df_normal = df[df[__TARGET].isin(['BENIGN', 'Normal'])]
    df_attack = df[df[__TARGET].isin(__LIST_ATK)]
    
    df_train_nor, df_test_nor = train_test_split(df_normal, test_size=rate, random_state=__SEED)
        
    dfs = []
    for name in __LIST_ATK:
        df_tmp = df_attack[df_attack[__TARGET] == name]  
        df_tmp = pd.concat([df_test_nor, df_tmp], axis=0)
        df_tmp[__TARGET] = df_tmp[__TARGET].apply(lambda x: 0 if (x == 'BENIGN' or x == 'Normal') else 1)
        dfs.append(df_tmp)
    
    
    return df_train_nor, pd.Series(data=dfs,index=__LIST_ATK)
    
def Normal_attack_split(df):
    global __UNSUP, __LIST_ATK
    __UNSUP = "UNSUP"
    __LIST_ATK = df[__TARGET].unique()
    
    __LIST_ATK = __LIST_ATK[ (__LIST_ATK != "BENIGN") & (__LIST_ATK != "Normal") ]

    df_normal = df[df[__TARGET].isin(['BENIGN', 'Normal'])]
    df_attack = df[df[__TARGET].isin(__LIST_ATK)]
    
    return df_normal, df_attack

In [216]:
__Atks_profile = {
    "CIC_IDS_2017":{
        # 'Malware': ['Bot']
    },
    "CIC_DDoS_2019":{
    },
    "BoT_IoT":{
        'Malware': ['Theft'],
        'DoS': ['DoS'],
        'DDoS': ['DDoS'],
        'Scan': ['Reconnaissance']
    },
    
    "ToN_IoT":{
        'Malware': ['backdoor','ransomware'],
        'Scan': ['scanning'],
        'BruteForce': ['password'],
        'DDoS': ['ddos'],
        'WebAttack': ['xss','injection'],
        'DoS': ['dos']
    },
    "N_BaIoT":{ # not map
        'Scan': ['scan'],
        'DoS': ['udp','tcp','syn','ack','udpplain','combo','junk'],
    },
    "UNSW_NB15":{
        'Malware': ['Exploits','Shellcode','Backdoor','Worms'],
        'DoS': ['DoS','Fuzzers'],
        'Scan': ['Reconnaissance','Analysis'],
    },
    "CIC_IoT2023": {
        'BruteForce': ['DictionaryBruteForce'],
        'Scan': ['Recon-PingSweep', 'Recon-OSScan', 'VulnerabilityScan', 'Recon-PortScan','Recon-HostDiscovery'],
        'WebAttack': ['SqlInjection', 'CommandInjection', 'Backdoor_Malware', 'Uploading_Attack','XSS','BrowserHijacking'],
        'Mirai': ['Mirai-greip_flood', 'Mirai-greeth_flood', 'Mirai-udpplain'],
    	'DDoS':['DDoS-ICMP_Flood', 'DDoS-UDP_Flood', 'DDoS-TCP_Flood', 'DDoS-SYN_Flood', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SynonymousIP_Flood',  'DDoS-ICMP_Fragmentation', 'DDoS-ACK_Fragmentation', 'DDoS-UDP_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-SlowLoris'],
    	'DoS': ['DoS-UDP_Flood', 'DoS-TCP_Flood', 'DoS-SYN_Flood', 'DoS-HTTP_Flood'],
    	'Web': ['Backdoor_Malware', 'Uploading_Attack' 'CommandInjection', 'XSS', 'SqlInjection', 'BrowserHijacking'],
    	'Spoofing': ['MITM-ArpSpoofing', 'DNS_Spoofing']
    }

}

_category_map = {
    'CIC_IoT2023': {
      '0Normal':            "0Normal",
      # 'DDoS-ACK_Fragmentation':   "DDoS",
      'DDoS-UDP_Flood':           "DDoS/DoS",
      # 'DDoS-SlowLoris':           "DDoS",
      'DDoS-ICMP_Flood':          "DDoS/DoS",
      # 'DDoS-RSTFINFlood':         "DDoS/DoS",
      # 'DDoS-PSHACK_Flood':        "DDoS",
      # 'DDoS-HTTP_Flood':          "DDoS",
      # 'DDoS-UDP_Fragmentation':   "DDoS",
      # 'DDoS-ICMP_Fragmentation':  "DDoS/DoS",
      'DDoS-TCP_Flood':           "DDoS/DoS",
      # 'DDoS-SYN_Flood':           "DDoS",
      # 'DDoS-SynonymousIP_Flood':  "DDoS",
        
      # 'DictionaryBruteForce':     "BruteForce",
      'MITM-ArpSpoofing':         'Spoofing',
      
        # 'DNS_Spoofing':             'Spoofing',
      'DoS-TCP_Flood':            'DDoS/DoS',
      # 'DoS-HTTP_Flood':           'DoS',
      # 'DoS-SYN_Flood':            'DoS',
      'DoS-UDP_Flood':            'DDoS/DoS',
      # 'Recon-PingSweep':          'Scan',
      # 'Recon-OSScan':             'Scan',
      
        'VulnerabilityScan':        'Scan',
      
        # 'Recon-PortScan':           'Scan',
      # 'Recon-HostDiscovery':      'Scan',
      # 'SqlInjection':             'Web_1',
      # 'CommandInjection':         'Web_2',
     
        'Backdoor_Malware':         'Web',
      # 'Uploading_Attack':         'Web_4',
      # 'XSS':                      'Web_5',
      # 'BrowserHijacking':         'Web_6',
      # 'Mirai-greip_flood':        'Mirai',
      # 'Mirai-greeth_flood':       'Mirai',
      
        'Mirai-udpplain':           'Mirai'
    },
    # 'N_BaIoT': {
    #   '0Normal':            "0Normal",
    #   'tcp'    :            "tcp"
    # },

    'BoT_IoT': { #map
        '0Normal': "0Normal",
        'Data_Exfiltration': "theft",
        'HTTP': "HTTP",
        'Keylogging': "theft",
        'OS_Fingerprint': "scan",
        'Service_Scan': "scan",
        'TCP': "TCP",
        'UDP': "UDP"
    },
    # ['udp' 'tcp' 'scan' 'syn' 'ack' 'udpplain' 'combo' 'junk' '0Normal']

    "UNSW_NB15":{ # not map nhưng bỏ 1 label Analysis 
        # '0Normal': "0Normal",
        # 'Exploits':'Malware',
        # 'Shellcode':'Malware',
        # 'Backdoor':'Malware',
        # 'Worms':'Malware',
        # 'DoS': 'DoS',
        # 'DoS': 'Fuzzers',
        # 'Reconnaissance':'Scan',
        # 'Analysis':'Scan'
        '0Normal': "0Normal",
        'Exploits':'Exploits',
        'Shellcode':'Shellcode',
        'Backdoor':'Backdoor',
        'Worms':'Worms',
        'DoS': 'DoS',
        'DoS': 'Fuzzers',
        'Reconnaissance':'Reconnaissance',
        'Analysis':'Analysis'
    },
    "ToN_IoT":{ 
        '0Normal': "0Normal",
        'backdoor': 'Malware',
        'ransomware': 'Malware',
        'scanning': 'Scan',
        'password': 'BruteForce',
        'ddos': 'DDoS',
        'xss': 'WebAttack',
        'injection':'WebAttack',
        'dos': 'DoS',
        'mitm': 'MITM'
    }
}

def load_n_baiot():
    data_dir = os.path.join(__DATA_DIR,"N_BaIoT")
    df_nors, df_anos = Import_dataset(data_dir, devices = __DEVICES, bot_type = 2, atk_type = "ALL")
    
    # dropping duplicate values 
    # df_nors.drop_duplicates(keep=False, inplace=True) 
    # df_anos.drop_duplicates(keep=False, inplace=True) 
    
    df_anos = Get_sample_ratio_by_column(df_anos, "Names Atk", __LIMIT_CNT)
    df_nors = Get_sample_ratio_by_column(df_nors, "Label", __LIMIT_CNT)
   
    df_anos = Preprocess_data(df_anos)
    df_nors = Preprocess_data(df_nors)
    df =  pd.concat([df_anos,df_nors], ignore_index=True).reset_index(drop=True)
    return df


def load_iotid20():
    data_dir = os.path.join(__DATA_DIR,"IoTID20")
    df = pd.read_csv(f"{data_dir}/iotid20.csv")
    # dropped_cols =["Flow_ID","Src_IP","Dst_IP","Timestamp"]
    # df.drop(columns=dropped_cols, inplace=True)
    # df.drop_duplicates(inplace=True, ignore_index=True)
    
    df = Get_sample_ratio_by_column(df, "Target", __LIMIT_CNT)    
   
    # df.drop(columns=dropped_cols, inplace=True, errors='ignore')
    df.rename(columns={'Target': __TARGET}, inplace=True)
    
    return df

def load_ids_by_name(name):
    df = None
    if name == "IoTID20":
        df = load_iotid20()
    else:
        if name == "N_BaIoT":
            df = load_n_baiot()
        else:
            df = load_data_ids(name, __DATA_DIR, __LIMIT_CNT)
    # df['Datasets'] = np.full(df.shape[0], name)
    df = df.drop(columns = ['Binary_dtloader','Category_dtloader'], errors='ignore')
    df[__TARGET] = df[__TARGET].apply(lambda x: "0Normal" if (x in ['BENIGN', 'Normal','normal','Benign','BenignTraffic']) else x)
    
    # df = df[df[__TARGET].isin(__Atks_profile[name][__Target_atk] + ["0Normal"])]

    
    # Bỏ comment cái này
    if name != "N_BaIoT" and name != "IoTID20":
        keys_list = list(_category_map[name].keys())
        df  = df[df[__TARGET].isin(keys_list)]
        
        df[__TARGET] = df[__TARGET].apply(lambda x: _category_map[name][x])
    return df


## Function to Create synthetic Data

In [217]:
from numpy import random as np_random
import random

def generate_multiclass_dataset(N, M, normal_points_per_cluster, n_features, anomaly_points):

    # Generate clusters
    if __DATA_TYPE == "MIX CLUSTER":
        data, labels = make_blobs(n_samples=N * normal_points_per_cluster, n_features = n_features,
                              centers=N, cluster_std=10.0, random_state=__SEED)
    if  __DATA_TYPE == "CLUSTER":
        data, labels = make_blobs(n_samples=M * normal_points_per_cluster, n_features = n_features,
                              centers=M, cluster_std=10.0, random_state=__SEED)
    if __DATA_TYPE == "MIX":
        data, labels = make_multilabel_classification(n_samples=M * normal_points_per_cluster, 
                                        n_features = n_features, allow_unlabeled = False,
                                  n_classes=M, random_state=__SEED)
        labels = np.argmax(labels, axis=1)

    if __DATA_TYPE == "MIX CLUSTER":
        data, labels = merge_and_encode_labels(data, labels, M)
    

    return data, labels


def merge_and_encode_labels(data, labels, M):
    """
    Merges labels randomly until M unique labels are left and then encodes these labels.

    :param data: 2D numpy array where each row is a sample.
    :param labels: 1D numpy array of labels corresponding to the data samples.
    :param M: The number of unique labels to be left after merging.
    :return: Updated data and labels.
    """
    unique_labels = np.unique(labels)
    n_unique = len(unique_labels)

    if M >= n_unique:
        print("M is greater than or equal to the number of unique labels. No merging needed.")
        return data, labels

    # Randomly merge labels
    while n_unique > M:
        # Pick two different labels randomly
        label1, label2 = random.sample(list(unique_labels), 2)

        # Merge label2 into label1
        labels[labels == label2] = label1

        # Update unique labels
        unique_labels = np.unique(labels)
        n_unique = len(unique_labels)

    # Label encoding
    encoder = LabelEncoder()
    encoded_labels = encoder.fit_transform(labels)

    return data, encoded_labels

def generate_random_array(N, S):
    """
    Generate a random array of size N with sum equal to S.

    Parameters:
    - N: Size of the output array.
    - S: Sum of all values in the output array.

    Returns:
    - random_array: Generated random array.
    """
    # Generate N-1 random values between 0 and S
    random_values = np.random.uniform(0, S, N-1)
    
    # Sort the random values and insert 0 at the beginning and S at the end
    random_values = np.sort(random_values)
    random_values = np.insert(random_values, 0, 0)
    random_values = np.append(random_values, S)
    
    # Compute the differences between consecutive values to get the array
    random_array = np.diff(random_values)

    return random_array

# This def to create perfect synthetic data

def generate_independently_vectors(n_cls, n_fts, n_sample_train, n_sample_test):

    if n_fts < n_sample_train:
        print("n_fts > n_sample_train")
        return None

    
    n_ = np.array([n_sample_train // n_cls for i in range(n_cls)])
    m_ = np.array([n_sample_test // n_cls for i in range(n_cls)])
    n_[n_cls-1] += (n_sample_train - (n_sample_train // n_cls) * n_cls)
    m_[n_cls-1] += (n_sample_test - (n_sample_test // n_cls) * n_cls)

    # if n_fts < n_sample_train:
    #     n_add = np.array([n_fts // n_cls for i in range(n_cls)])
    #     n_add[n_cls-1] += (n_fts - (n_fts // n_cls) * n_cls)
    #     n_add = n_add - n_
    # else:
    #     n_add = np.zeros(n_sample_train)
    
    X_train = np.zeros((n_sample_train,n_fts))
    y_train = np.zeros(n_sample_train)

    X_test = np.zeros((n_sample_test,n_fts))
    y_test = np.zeros(n_sample_test)

    id = 0
    id_t = 0
    for k in range(n_cls):
        x_cls = np.zeros(n_[k])
        for i in range(n_[k]):
            X_train[id+i][id+i] = np_random.randint(1,3)
            y_train[id+i] = k
       

        for i in range(m_[k]):
            tmp = generate_random_array(n_[k],1)
            for t in range(n_[k]):
                X_test[id_t + i][id+t] = tmp[t] * X_train[id+t][id+t]
                # X_test[id_t + i][id+t] = tmp[t]
                # res_tmp = tmp[t]
                # for x in range(n_[k]):
                    # res_tmp = res_tmp + res_tmp*X_train[id+x][id+x]
                    
                # X_test[id_t + i][id+t] = res_tmp
                
            y_test[id_t+i] = k
            
        id = id + n_[k]
        id_t = id_t + m_[k]
        
    return X_train, X_test, y_train, y_test






## Set Logger

In [218]:
def setup_logger(log_file='example.log', level=logging.DEBUG):
    # Create a logger
    logger = logging.getLogger('my_logger')
    logger.setLevel(level)

    # Create a file handler and set the level to the specified level
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(level)

    # Create a formatter and add it to the file handler
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)

    # Add the file handler to the logger
    logger.addHandler(file_handler)

    return logger




## Function to Train/Test Model machine learing

In [219]:
def Split_Xy(df):
  y = df[__TARGET]
  X = df.drop(columns=[__TARGET])
  return X.to_numpy(),y.to_numpy()

def Get_Scaler(name):
  # (StandardScaler, MinMaxScaler, RobustScaler, Normalizer)
  if name == "StandardScaler":
    return StandardScaler()
  if name == "MinMaxScaler":
    return MinMaxScaler()
  if name == "RobustScaler":
    return RobustScaler()
  if name == "Normalizer":
    return Normalizer()
  if name == "QuantileTransformer":
      return QuantileTransformer(output_distribution = "normal", random_state=__SEED)
  return None
  
def ML_train(X=None, y = None, model=None):
    t_start = time.time()

    if y is not None:
        model.fit(X,y)
    else:
        model.fit(X)

    t_end = time.time()-t_start

    return model, t_end

def ML_test(X = None, y = None, model=None):

    t_start = time.time()

    Xs = np.array_split(X, int(X.shape[0] / 10000) + 1)
   
    y_pred = np.array([])
    
    for X_tmp in Xs:
        y_pred_tmp = model.predict(X_tmp)
    
        y_pred = np.concatenate((y_pred,y_pred_tmp))


    t_end = time.time()-t_start

    return y, y_pred, t_end

def Model_evaluating(y_true, y_pred, __Label_use_name, export_fig = None):
    import seaborn as sns
    import matplotlib.pyplot as plt

    emberdf = pd.DataFrame({
        'y_pred': y_pred.astype(int),
        'label': y_true.astype(int)})

    mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, cnf_matrix = calc_index(emberdf,__Label_use_name,export_fig)
    clf_report = classification_report(emberdf.label,
                                    emberdf.y_pred,
                                    labels=[*range(len(__Label_use_name))],
                                    target_names=__Label_use_name,
                                    output_dict=True,
                                    zero_division = 0)
    # logger.info("Classification report:")
    # logger.info(classification_report(emberdf.label,
    #                                     emberdf.y_pred,
    #                                     labels=[*range(len(__Label_use_name))],
    #                                     target_names=__Label_use_name,
    #                                     output_dict=False,
    #                                     zero_division = 0))
    # clf_report = []
  # plot cfs report
  # fig = sns.heatmap(pd.DataFrame(clf_report).iloc[:-1, :].T, annot=True).get_figure()  
  # plt.show()

    if export_fig is not None:
        fig.savefig(os.path.join(export_fig,'cfs_report.png'))
  
  # print("=========================================================")  
            
    return mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix

# KNFST

In [220]:
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.metrics import classification_report


import numpy as np
from sklearn.metrics.pairwise import pairwise_kernels


def K(X,Y=None,metric='poly',coef0=1,gamma=None,degree=3):
    if metric == 'poly':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    elif metric == 'linear':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    elif metric == 'sigmoid':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    elif metric == 'rbf':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    return k

def kernel_distance_matrix(X=None, kernel="linear"):
    """
    Calculate the distance matrix using the kernel trick.

    Parameters:
    - X: Input data, a 2D numpy array where each row represents a sample.
    - kernel: Kernel function. Default is __KERNEL.

    Returns:
    - distance_matrix: Distance matrix.
    """
    # Calculate kernel matrix
    kernel_matrix = K(X, metric=kernel)

    # Calculate distance matrix using the given formula
    diagonal = np.diagonal(kernel_matrix)
    distance_matrix = np.outer(diagonal, np.ones(X.shape[0])) - 2 * kernel_matrix + np.outer(np.ones(X.shape[0]), diagonal)

    return distance_matrix

def kernel_distance(matrix1, matrix2, kernel="linear"):
    """
    Calculate the distance between two matrices using the kernel trick.

    Parameters:
    - matrix1: The first input matrix (NumPy array).
    - matrix2: The second input matrix (NumPy array).
    - gamma: The gamma parameter for the RBF kernel.

    Returns:
    - distance_matrix: The distance matrix between the two input matrices.
    """
    if matrix1.shape[1] != matrix2.shape[1]:
        raise ValueError("The number of features in the input matrices must be the same.")
        
    Kaa = []
    for i in range(len(matrix1)):
        Kaa.append(K(matrix1[i,:].reshape(1,-1),metric=kernel))    
    Kaa = np.asarray(Kaa).ravel().reshape(len(Kaa),1)
    
    Kab = K(matrix1,matrix2,metric=kernel)
    Kbb = []
    for i in range(len(matrix2)):
        Kbb.append(K(matrix2[i,:].reshape(1,-1),metric=kernel))
    Kbb = np.asarray(Kbb).ravel()
    
    d = Kaa-2*Kab+Kbb #shape: (matrix1,matrix2)

    return d

In [221]:
from scipy.linalg import svd
from scipy.special import softmax

def nullspace(A, eps=1e-12):
    u, s, vh = svd(A)
    null_mask = (s <= eps)
    null_space = sp.compress(null_mask, vh, axis=0)
    return sp.transpose(null_space)

from scipy.sparse import csr_matrix

def compute_sparse_L(labels, classes):
    n = len(labels)
    classes, counts = np.unique(labels, return_counts=True)
    inverse_counts = 1.0 / counts
    class_to_inverse_count = dict(zip(classes, inverse_counts))

    # Create a vector 'inverse_count_vector' where each entry is 1/counts[class] for the class of the corresponding label
    inverse_count_vector = np.vectorize(class_to_inverse_count.get)(labels)

    # Now create the sparse matrix L using this vector
    # The rows and cols correspond to all pairs of indices
    rows, cols = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')

    # Only include positions where the labels are equal
    mask = labels[rows] == labels[cols]
    rows = rows[mask]
    cols = cols[mask]
    data = inverse_count_vector[rows]

    # Create the csr_matrix
    L_sparse = csr_matrix((data, (rows, cols)), shape=(n, n))

    return L_sparse

def learn(K, labels):
    classes = np.unique(labels)
    if len(classes) < 2:
        raise Exception("KNFST requires 2 or more classes")
    n, m = K.shape
    if n != m:
        raise Exception("Kernel matrix must be quadratic")

    centered_k = KernelCenterer().fit_transform(K)

    basis_values, basis_vecs = np.linalg.eig(centered_k)
    
    basis_vecs = basis_vecs[:, basis_values > 1e-12]
    basis_values = basis_values[basis_values > 1e-12]

    basis_values = np.diag(1.0 / np.sqrt(basis_values))

    basis_vecs = basis_vecs.dot(basis_values)


    classes = np.unique(labels)
    
    
    L = compute_sparse_L(labels, classes)

    M = np.ones([m, m]) / m
    H = (((np.eye(m, m) - M).dot(basis_vecs)).T).dot(K).dot(np.eye(n, m) - L)

    # print("H:", H.shape)

    t_sw = H.dot(H.T)
    eigenvecs = nullspace(t_sw) 
    # print("eigenvecs:", eigenvecs.shape)

    if eigenvecs.shape[1] < 1:
        eigenvals, eigenvecs = np.linalg.eigh(t_sw)

        eigenvals = np.diag(eigenvals)
        min_idx = eigenvals.argsort()[0]
        eigenvecs = eigenvecs[:, min_idx]


    proj = ((np.eye(m, m) - M).dot(basis_vecs)).dot(eigenvecs)
    
    target_points = []
    for cl in classes:
        k_cl = K[labels == cl, :]
        pt = np.mean(k_cl.dot(proj), axis=0)

        target_points.append(pt)

    return proj, np.array(target_points)

def squared_euclidean_distances(x, y):
    n = np.shape(x)[0]
    m = np.shape(y)[0]
    distmat = np.zeros((n,m))
    
    for i in range(n):
        for j in range(m):
            buff = x[i,:] - y[j,:]
            distmat[i,j] = buff.dot(buff.T)
    return distmat

def assign_score(proj, target_points, ks):
    projection_vectors = ks.T.dot(proj)

    # New distance
    target_points = target_points.real
    projection_vectors = projection_vectors.real
    scores = np.sqrt(kernel_distance(projection_vectors, target_points, __KERNEL))

    return scores


# select the largetest element that is smaller than the threshold in a list
def select_largest_smaller_than_threshold(threshold, list):
    for i in range(len(list)-1, -1, -1):
        if list[i] < threshold:
            return i
    return -1

In [222]:
# a = np.array([[[1,2,3]],[[11,2,3]]])
# print(len(a.shape))


In [223]:
def train_knfst(X_train, y_train, learn):
    kernel_mat = metrics.pairwise_kernels(X_train, metric=__KERNEL)
    # target_points là các điểm trung bình của các class trong mặt phẳng nullspace
    proj, target_points = learn(kernel_mat, y_train)
    if len(target_points.shape) == 3:
        target_points = np.reshape(target_points,(target_points.shape[0],target_points.shape[2]))
        proj = np.array(proj)
    return proj, target_points

def test_knfst_novelty(proj, target_points, X_train, X_test, y_test, encoder):
    t0 = time.time()
    ks = metrics.pairwise_kernels(X_train, X_test, metric=__KERNEL)
    # __Label_use_name = np.char.mod('%d', np.unique(y_test))
    if __MODE == "Novel":
        __Label_use_name =  ["-1"] + list(encoder.classes_)
    else:
        __Label_use_name =  list(encoder.classes_)
        
    print(f"__Label_use_name: {__Label_use_name}")

    # print(type(X_train))
    # print(type(X_test))
    # print(type(proj))
    # print(type(target_points))
    # print(type(ks))

    # print(X_train.shape)
    # print(X_test.shape)
    # print(proj.shape)
    # print(target_points.shape)
    # print(ks.shape)
    
    # Khoảng cách từ sample tới mỗi điểm trung bình của các class
    # Score càng bé, khoảng cách càng nhỏ -> càng gần với class đó
    scores_mat = assign_score(proj, target_points, ks)
    
    y_pred = np.argmin(scores_mat, axis=1)
        
    # Novelty scores bé nhất -> lớn hơn threshold -> là novel
    scores = np.amin(scores_mat, axis=1)

    t1 = time.time()
    
    if __MODE == "Unsup":
        y_pred_bina = []
        y_pred_adv = y_pred
        threshold = -1
        ndr = -1
        auc_n = -1
    else:
        y_pred_adv, mcc, threshold = Bruteforce_threshold(y_test, y_pred, scores)
        
        y_test_bina = np.array([1 if y == -1 else 0 for y in y_test])
        y_pred_bina = np.array([1 if y == -1 else 0 for y in y_pred_adv])
        

        ndr = recall_score(y_test_bina,y_pred_bina,zero_division = 0)
        false_positive_rate, true_positive_rate, _ = roc_curve(y_test_bina, y_pred_bina)
        auc_n = auc(false_positive_rate, true_positive_rate)
        # auc_n = -1

    mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix = Model_evaluating(y_test, y_pred_adv, __Label_use_name,None)

    # logger.info(f"Matthews corrcoef score: {mcc}")
 

    # print("Classification report")
    # # print(clf_report)
    # print(classification_report(y_test, y_pred_adv,
    #                                     labels=np.unique(y_test),
    #                                     target_names=__Label_use_name,
    #                                     output_dict=False,
    #                                     zero_division = 0))

    return mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix, y_pred_adv, y_pred_bina, scores, threshold, t1-t0

In [224]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def visualize_scaled_distribution(data):
    """
    Visualize the distribution of data after scaling it to a normal distribution using Z-score normalization.

    Parameters:
    - data: Input NumPy array.

    Returns:
    - None (displays the histogram plot).
    """
    # Apply Z-score normalization to scale the data to a normal distribution
    scaled_data = stats.zscore(data)

    # Create a histogram of the scaled data
    plt.hist(scaled_data, bins=20, color='skyblue', edgecolor='black')

    # Add labels and title
    plt.title('Distribution of Scaled Data (Normal Distribution)')
    plt.xlabel('Scaled Data Values')
    plt.ylabel('Frequency')

    # Display the plot
    plt.show()




# HHH

In [225]:
class HHH:
    def __init__(self, kernel=None):
        """
        Init the model.

        Parameters:
        
        - kernel: kernel used, support: "linear", "poly", "rbf", "sigmoid"

        """
        self.X_train = None
        self.y_train = None
        self.kernel = None
        self.projec_vec = None
        self.y_preds = None
        self.y_scores = None

    def _gaussian_elimination(self, vectors, threshold=1e-10):
        """Performs Gaussian elimination while preserving the order of vectors.
    
        Args:
            vectors: A numpy array where each row represents a vector.
            threshold: A value below which numbers are considered zero for numerical stability.
    
        Returns:
            Two lists:
                - A list of linearly independent vectors in their original order.
                - A list of indices indicating the positions of the linearly independent vectors
                in the original input array.
        """
    
        matrix = vectors.copy()  # Ensure original data isn't modified
        num_rows, num_cols = matrix.shape
        independent_indices = []
    
        for col in range(num_cols):
            # Find the pivot row (the row with the largest absolute value in the current column)
            pivot_row = np.argmax(np.abs(matrix[col:, col])) + col  # Vectorized operation
    
            # If the pivot element is close to zero, skip this column
            if abs(matrix[pivot_row, col]) < threshold:
                continue
    
            # Swap the current row with the pivot row ONLY IF THEY ARE DIFFERENT
            if pivot_row != col:
                matrix[[col, pivot_row]] = matrix[[pivot_row, col]]
    
            # Normalize the pivot row and eliminate elements below
            pivot = matrix[col, col]
            matrix[col] /= pivot
    
            # Eliminate the elements below the pivot (vectorized operation)
            rows_below = matrix[col+1:, col]  
            matrix[col+1:] -= np.outer(rows_below, matrix[col])
    
            independent_indices.append(pivot_row)
    
        # Extract linearly independent vectors (non-zero rows) in their original order
        independent_vectors = vectors[independent_indices]  # Direct indexing
    
        return independent_vectors, independent_indices

    
    def _conjugate_grad(self, A, b, x=None):
        """
        Description
        -----------
        Solve a linear equation Ax = b with conjugate gradient method.
        Parameters
        ----------
        A: 2d numpy.array of positive semi-definite (symmetric) matrix
        b: 1d numpy.array
        x: 1d numpy.array of initial point
        Returns
        -------
        1d numpy.array x such that Ax = b
        """
        # n = len(b)
        # if x is None:
        #     x = np.ones(n)
        
        # r = np.dot(A, x) - b
        # p = - r
        # r_k_norm = np.dot(r, r)
        
        # for i in trange(3*n):
            
        #     Ap = np.dot(A, p)
            
        #     alpha = r_k_norm / np.dot(p, Ap)
            
        #     x += alpha * p
        #     r += alpha * Ap
            
        #     r_kplus1_norm = np.dot(r, r)
            
        #     beta = r_kplus1_norm / r_k_norm
            
        #     r_k_norm = r_kplus1_norm
    
        #     print(f"Current iter {i} , loss: {r_kplus1_norm}", end='\r')
        #     if r_kplus1_norm < 1e-12:
        #         print( 'Itr:', i)
        #         break
            
        #     p = beta * p - r

        # using sparse matrix
        A = csr_matrix(A)
        # print("DEBUG - CHECK 1")
        
        # Solve the linear system Ax = b for x
        x = spsolve(A, b)
        # print("DEBUG - CHECK 2")
        
        return x
    
    def _without_kernel_train(self):
        n = self.X_train.shape[0]
        Q = self.X_train.T
        b = self.y_train.T
        
        # KO xaif Kernel
        A = np.dot(Q.T,Q) + np.eye(n,n) * 1e-5
    
        alpha = self._conjugate_grad(A, b)
        theta = np.dot(Q,alpha)
        
        return theta


    def _kernel_train(self):
        n = self.X_train.shape[0]
        
        b = self.y_train.T
        
        A = metrics.pairwise_kernels(self.X_train, metric=self.kernel) + np.eye(n,n) * 1e-5
        
        alpha = self._conjugate_grad(A, b)
        
    
        return alpha

    def _without_kernel_transform(self, X):
        X_res = []
        theta = self.projec_vec
        for x in X:
            X_res.append(x * theta)
            
        return np.array(X_res)

    def _kernel_transform(self, X):
    
        score_matrix = metrics.pairwise_kernels(X, self.X_train, metric=self.kernel)
    
        X_res = []
        alpha = self.projec_vec
        
        for i in range(0, len(X)):
            X_res.append(score_matrix[i] * alpha)
                  
        return np.array(X_res)
        
    def _without_kernel_predict(self, X_test):
        # kernel_distance
        y_pred = []
        y_score = []
        theta = self.projec_vec
        max_p = max(self.y_train)
        min_p = min(self.y_train)
        
        for x in X_test:
            score = np.sum(x * theta)
            y_score.append(score)
            pred = round(score)
            pred = pred if (pred <= max_p and pred>=min_p) else -1
            y_pred.append(pred)
            
        return np.array(y_pred), np.array(y_score)

    def _kernel_predict(self, X_test):
        
        # score_matrix = kernel_distance(X_train, X_test, __KERNEL)
        score_matrix = metrics.pairwise_kernels(X_test, self.X_train, metric=self.kernel)
    
        y_pred = []
        y_score = []
        alpha = self.projec_vec
        max_p = max(self.y_train)
        min_p = min(self.y_train)
        
        for i in range(0, len(X_test)):
            score = np.sum(score_matrix[i] * alpha)
            y_score.append(score)
            pred = round(score)
            pred = pred if (pred <= max_p and pred>=min_p) else -1
            y_pred.append(pred)
        
        return np.array(y_pred), np.array(y_score)
    
    def _form_independent(self, X_train, y_train):
        dataX = []
        datay = []
        labels = np.unique(y_train)
        print("Debug ===== labels:", labels)
        for label in labels:
            mask = np.array([y == label for y in y_train])
            data = X_train[mask]
            if self.kernel is not None:
                data = metrics.pairwise_kernels(data, metric=self.kernel)
                data = np.array(data)
            _, indicates = self._gaussian_elimination(data)
            # print(type(data), data.shape)
            dataX = dataX + X_train[mask][indicates].tolist()
            datay = datay + [label]*len(indicates)
        print("Debug ===== dataX:", np.array(dataX).shape)
        return np.array(dataX), np.array(datay)
        
    def fit(self, X_train = None, y_train = None, kernel = None):
        
        self.kernel = kernel
 
        dataX_train, datay_train = self._form_independent(X_train, y_train)
        # dataX_train = X_train
        # datay_train = y_train
        self.X_train = dataX_train
        self.y_train = datay_train
        

        if self.kernel is None:
            self.projec_vec = self._without_kernel_train()
        else:
            self.projec_vec = self._kernel_train()

    def transform(self, X = None):
        if self.projec_vec is None:
            print("Error: Fit data first")
            return X
    
        if self.kernel is None:
            X = self._without_kernel_transform(X)
        else:
            X = self._kernel_transform(X)
            
        return X
            
    def predict(self, X_test = None):
        if self.kernel is None:
            self.y_preds, self.y_scores = self._without_kernel_predict(X_test)
        else:
            self.y_preds, self.y_scores = self._kernel_predict(X_test)

        return self.y_preds, self.y_scores
        
    def evaluating(self, X_test = None, y_test = None, __Label_use_name = None , export_fig = None):

        if self.projec_vec is None:
            print("Error: Training model first")
            return None

        if y_test is None:
            print("Error: Provided y_test")
            return None
            
        if X_test is None:
            print("Error: Provided X_test")
            return None

        self.predict(X_test)
        
        emberdf = pd.DataFrame({
            'y_pred': y_test.astype(int),
            'label': self.y_preds.astype(int)})
        
        mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, cnf_matrix = calc_index(emberdf,__Label_use_name,export_fig)

        if "-1" in __Label_use_name:
            y_test_bina = np.array([1 if y == -1 else 0 for y in y_test])
            y_pred_bina = np.array([1 if y == -1 else 0 for y in self.y_preds])
            
    
            ndr = recall_score(y_test_bina,y_pred_bina,zero_division = 0)
            false_positive_rate, true_positive_rate, _ = roc_curve(y_test_bina, y_pred_bina)
            auc_n = auc(false_positive_rate, true_positive_rate)
            # auc_n = -1
        else:
            ndr = -1
            auc_n = -1

        
        clf_report = classification_report(emberdf.label,
                                        emberdf.y_pred,
                                        labels=[*range(len(__Label_use_name))],
                                        target_names=__Label_use_name,
                                        output_dict=True,
                                        zero_division = 0)
        # print("Classification report:")
        # print(classification_report(emberdf.label,
        #                                     emberdf.y_pred,
        #                                     labels=[*range(len(__Label_use_name))],
        #                                     target_names=__Label_use_name,
        #                                     output_dict=False,
        #                                     zero_division = 0))
            
        return mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix
        
        
        

# HHHv2

In [226]:
def softmax(x):
    """Compute softmax values for each set of scores in x."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)
    
# enc = OneHotEncoder()
# a = np.array([1,2,3,1,2,3]).reshape(-1, 1)
# b = enc.fit_transform(a).toarray()
# print(b.shape)

In [227]:
class HHHv2:
    def __init__(self, kernel=None):
        """
        Init the model.

        Parameters:
        
        - kernel: kernel used, support: "linear", "poly", "rbf", "sigmoid"

        """
        self.X_train = None
        self.y_train = None
        self.kernel = None
        self.projec_vec = None
        self.y_preds = None
        self.y_scores = None

    # def _gaussian_elimination(self, vectors, threshold=1e-10):
    #     """Performs Gaussian elimination while preserving the order of vectors.
    
    #     Args:
    #         vectors: A numpy array where each row represents a vector.
    #         threshold: A value below which numbers are considered zero for numerical stability.
    
    #     Returns:
    #         Two lists:
    #             - A list of linearly independent vectors in their original order.
    #             - A list of indices indicating the positions of the linearly independent vectors
    #               in the original input array.
    #     """
    
    #     matrix = vectors.copy()
    #     num_rows, num_cols = matrix.shape
    #     independent_indices = []  # Track indices of independent vectors
    
    #     row = 0
    #     for col in range(num_cols):
    #         # Find the pivot row (the row with the largest absolute value in the current column)
    #         pivot_row = row
    #         for i in range(row + 1, num_rows):
    #             if abs(matrix[i, col]) > abs(matrix[pivot_row, col]):
    #                 pivot_row = i
    
    #         # If the pivot element is close to zero, skip this column (the vector is linearly dependent)
    #         if abs(matrix[pivot_row, col]) < threshold:
    #             continue
    
    #         # Swap the current row with the pivot row ONLY IF THEY ARE DIFFERENT
    #         if pivot_row != row:
    #             matrix[[row, pivot_row]] = matrix[[pivot_row, row]]
    
    #         # Normalize the pivot row (make the pivot element equal to 1)
    #         pivot = matrix[row, col]
    #         matrix[row] /= pivot
    
    #         # Eliminate the elements below the pivot
    #         for i in range(row + 1, num_rows):
    #             factor = matrix[i, col]
    #             matrix[i] -= factor * matrix[row]
    
    #         independent_indices.append(pivot_row)  # Store the original index
    #         row += 1
    
    #     # Extract linearly independent vectors (non-zero rows) in their original order
    #     independent_vectors = [vectors[i] for i in independent_indices]
    
    #     return independent_vectors, independent_indices

    def _gaussian_elimination(self, vectors, threshold=1e-10):
        """Performs Gaussian elimination while preserving the order of vectors.
    
        Args:
            vectors: A numpy array where each row represents a vector.
            threshold: A value below which numbers are considered zero for numerical stability.
    
        Returns:
            Two lists:
                - A list of linearly independent vectors in their original order.
                - A list of indices indicating the positions of the linearly independent vectors
                in the original input array.
        """
    
        matrix = vectors.copy()  # Ensure original data isn't modified
        num_rows, num_cols = matrix.shape
        independent_indices = []

        max_rank = min(num_rows, num_cols)
    
        for col in range(max_rank):
            # Find the pivot row (the row with the largest absolute value in the current column)
            pivot_row = np.argmax(np.abs(matrix[col:, col])) + col  # Vectorized operation
    
            # If the pivot element is close to zero, skip this column
            if abs(matrix[pivot_row, col]) < threshold:
                continue
    
            # Swap the current row with the pivot row ONLY IF THEY ARE DIFFERENT
            if pivot_row != col:
                matrix[[col, pivot_row]] = matrix[[pivot_row, col]]
    
            # Normalize the pivot row and eliminate elements below
            pivot = matrix[col, col]
            matrix[col] /= pivot
    
            # Eliminate the elements below the pivot (vectorized operation)
            rows_below = matrix[col+1:, col]  
            matrix[col+1:] -= np.outer(rows_below, matrix[col])
    
            independent_indices.append(pivot_row)
    
        # Extract linearly independent vectors (non-zero rows) in their original order
        independent_vectors = vectors[independent_indices]  # Direct indexing
    
        return independent_vectors, independent_indices
    
    
    def _conjugate_grad(self, A, b, x=None):
        """
        Description
        -----------
        Solve a linear equation Ax = b with conjugate gradient method.
        Parameters
        ----------
        A: 2d numpy.array of positive semi-definite (symmetric) matrix
        b: 1d numpy.array
        x: 1d numpy.array of initial point
        Returns
        -------
        1d numpy.array x such that Ax = b
        """


    
        if b.shape[1] > 9000:
            C, n = b.shape
        
            x_res = []
            for i in range(C):
                x = np.ones(n)
                r = np.dot(A, x) - b[i]
                p = - r
                r_k_norm = np.dot(r, r)
                
                for i in trange(40000):
                    
                    Ap = np.dot(A, p)
                    
                    alpha = r_k_norm / np.dot(p, Ap)
                    
                    x += alpha * p
                    r += alpha * Ap
                    
                    r_kplus1_norm = np.dot(r, r)
                    
                    beta = r_kplus1_norm / r_k_norm
                    
                    r_k_norm = r_kplus1_norm
            
                    # print(f"Current iter {i} , loss: {r_kplus1_norm}", end='\r')
                    if r_kplus1_norm < 1e-12:
                        # print( 'Itr:', i)
                        break
                    
                    p = beta * p - r
                x_res.append(x)
            x_res = np.array(x_res).T
            return x_res

        # solve directly
        C = b.shape[0] #shape cxn
        x = []
        A = csr_matrix(A)
        for i in range(C):
            x.append(spsolve(A, b[i]))
        x = np.array(x).T
        
        return x
    
    def _without_kernel_train(self):
        n = self.X_train.shape[0]
            
        Q = self.X_train.T
        b = self.y_train.T
        
        # KO xaif Kernel
        A = np.dot(Q.T,Q) + np.eye(n,n) * 1e-5
    
        alpha = self._conjugate_grad(A, b)
        theta = np.dot(Q,alpha)
        
        return theta



    def _kernel_train(self):
        n = self.X_train.shape[0]
        
        b = self.y_train.T
        
        A = metrics.pairwise_kernels(self.X_train, metric=self.kernel) + np.eye(n,n) * 1e-5
        
        alpha = self._conjugate_grad(A, b)

    
        return alpha
        
    def _without_kernel_transform(self, X):
        X_res = []
        theta = self.projec_vec
        for x in X:
            X_res.append(np.dot(x.reshape(1,-1),theta).reshape(-1))
            
        return np.array(X_res)

    def _kernel_transform(self, X):
    
        score_matrix = metrics.pairwise_kernels(X, self.X_train, metric=self.kernel)
    
        X_res = []
        alpha = self.projec_vec
        
        for i in range(0, len(X)):
            X_res.append(np.dot(score_matrix[i].reshape(1,-1),alpha).reshape(-1))
                  
        return np.array(X_res)
    
        
    def _without_kernel_predict(self, X_test):
        # kernel_distance
        y_pred = []
        y_score = []
        theta = self.projec_vec
       
        for x in X_test:
            score = abs(np.dot(x.reshape(1,-1),theta) - 1)
            y_score.append(score.reshape(-1))
            pred = np.argmin(score)
            y_pred.append(pred)
  
            
        return np.array(y_pred), np.array(y_score)

    def _kernel_predict(self, X_test):
        
        score_matrix = metrics.pairwise_kernels(X_test, self.X_train, metric=self.kernel)
    
        y_pred = []
        y_score = []
        alpha = self.projec_vec
        
        for i in range(0, len(X_test)):
            score = abs(np.dot(score_matrix[i].reshape(1,-1),alpha) - 1)
            y_score.append(score.reshape(-1))
            pred = np.argmin(score)
            y_pred.append(pred)
        
        return np.array(y_pred), np.array(y_score)

    def _form_independent(self, X_train, y_train):
        dataX = []
        datay = []
        labels = np.unique(y_train)
        print("Debug ===== labels:", labels)
        for label in labels:
            mask = np.array([y == label for y in y_train])
            data = X_train[mask]
            if self.kernel is not None:
                data = metrics.pairwise_kernels(data, metric=self.kernel)
                data = np.array(data)
            _, indicates = self._gaussian_elimination(data)
            # print(type(data), data.shape)
            dataX = dataX + X_train[mask][indicates].tolist()
            datay = datay + [label]*len(indicates)
        print("Debug ===== dataX:", np.array(dataX).shape)
        return np.array(dataX), np.array(datay)
        
    def fit(self, X_train = None, y_train = None, kernel = None):
        
        self.kernel = kernel
 
        # dataX_train, datay_train = self._form_independent(X_train, y_train)
        # dataX_train = X_train
        # datay_train = y_train
        self.X_train = X_train
        self.y_train = y_train
        

        if len(self.y_train.shape) == 1:
            enc = OneHotEncoder()
            self.y_train = self.y_train.reshape(-1, 1)
            self.y_train = enc.fit_transform(self.y_train).toarray()

        if self.kernel is None:
            self.projec_vec = self._without_kernel_train()
        else:
            self.projec_vec = self._kernel_train()

    def transform(self, X = None):
        if self.projec_vec is None:
            print("Error: Fit data first")
            return X
    
        if self.kernel is None:
            X = self._without_kernel_transform(X)
        else:
            X = self._kernel_transform(X)
            
        return X
        
    def predict(self, X_test = None):
        if self.kernel is None:
            self.y_preds, self.y_scores = self._without_kernel_predict(X_test)
        else:
            self.y_preds, self.y_scores = self._kernel_predict(X_test)

        return self.y_preds, self.y_scores
        
    def evaluating(self, X_test = None, y_test = None, __Label_use_name = None , export_fig = None):

        if self.projec_vec is None:
            print("Error: Training model first")
            return None

        if y_test is None:
            print("Error: Provided y_test")
            return None
            
        if X_test is None:
            print("Error: Provided X_test")
            return None

        self.predict(X_test)
        
        emberdf = pd.DataFrame({
            'y_pred': y_test.astype(int),
            'label': self.y_preds.astype(int)})
        
        mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, cnf_matrix = calc_index(emberdf,__Label_use_name,export_fig)

        if "-1" in __Label_use_name:
            y_test_bina = np.array([1 if y == -1 else 0 for y in y_test])
            y_pred_bina = np.array([1 if y == -1 else 0 for y in self.y_preds])
            
    
            ndr = recall_score(y_test_bina,y_pred_bina,zero_division = 0)
            false_positive_rate, true_positive_rate, _ = roc_curve(y_test_bina, y_pred_bina)
            auc_n = auc(false_positive_rate, true_positive_rate)
            # auc_n = -1
        else:
            ndr = -1
            auc_n = -1

        
        clf_report = classification_report(emberdf.label,
                                        emberdf.y_pred,
                                        labels=[*range(len(__Label_use_name))],
                                        target_names=__Label_use_name,
                                        output_dict=True,
                                        zero_division = 0)
        # print("Classification report:")
        # print(classification_report(emberdf.label,
        #                                     emberdf.y_pred,
        #                                     labels=[*range(len(__Label_use_name))],
        #                                     target_names=__Label_use_name,
        #                                     output_dict=False,
        #                                     zero_division = 0))
            
        return mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix
        
        
        

# Test Combination in Classification mode

## Multiclass Novelty Detection

In [228]:
def prepare_data(data, target, cls_drop):
    classes = np.unique(target)
    if __MODE == "Novel":
        mask = ~np.isin(classes, cls_drop)
        known = classes[mask]
    else:
        known = classes

    print(known)

   
    data_train, data_test, target_train, target_test = train_test_split(data, target, test_size=0.2, stratify = target, random_state=__SEED)

    # Loại bỏ các class không biết trong tập train
    mask = np.array([y in known for y in target_train])
    
    X_train = data_train[mask]
    y_train = target_train[mask]

    idx = y_train.argsort()
    X_train = X_train[idx]
    y_train = y_train[idx]

    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)

    X_test = data_test
    y_labels = target_test

    if __MODE == "Novel":
        # Test labels are 1 if novel, otherwise 0.
        # y_test_bina = np.array([1 if cl not in known else 0 for cl in y_labels])
        y_test = np.array([-1 if cl not in known else encoder.transform([cl])[0] for cl in y_labels])
        # y_test = np.array([cl+"-1" if cl not in known else cl for cl in y_labels])
        
    
    if __MODE == "Unsup":
        # y_test_bina = np.array([1 if cl != 0 else 0 for cl in y_labels])
        y_test = encoder.transform(y_labels)
        
    # encoder = LabelEncoder()
    # y_test = encoder.fit_transform(y_test)
    # y_train = encoder.transform(y_train)

    classes = np.unique(y_train)
    # print("Final dataset size:", X_train.shape())

    return X_train, y_train, X_test, y_test, classes, encoder

def reduce_trainning_data(data, label):
    _, data_train, _, target_train = train_test_split(data, label, test_size=0.5, stratify = label, random_state=__SEED)
    return data_train, target_train

def calculate_accuracy_for_label(y_true, y_predict, label):
    """
    Calculate accuracy for a specific label.

    Parameters:
    - y_true: The true labels (1D NumPy array).
    - y_predict: The predicted labels (1D NumPy array).
    - label: The specific label for which to calculate accuracy.

    Returns:
    - accuracy: The accuracy for the specified label.
    """
    # Create a boolean mask for the specified label
    mask = (y_true == label)

    # Extract true labels and predicted labels for the specified label
    true_labels_for_label = y_true[mask]
    predicted_labels_for_label = y_predict[mask]

    # Calculate accuracy for the specified label
    accuracy = np.mean(true_labels_for_label == predicted_labels_for_label)

    return accuracy

def Bruteforce_threshold(y_test, y_pred, scores):
    
    min_th = 1e-7
    max_th = 1e-1
    # __step = int(max_th / min_th)
    __step = 100000
    mcc = matthews_corrcoef(y_test, y_pred)
    ndr = 0
    y_pred_adv = y_pred


    __rag = np.unique(y_test)
    thr = -np.ones(len(__rag))

    for id in __rag[1:]:
        # print("DEBUG:", id)
        for x in np.linspace(min_th, max_th, num=__step):

            y_pred_adv_tmp = np.array([-1 if (y_p == id) and (sc > x) else y_p for y_p, sc in zip( y_pred,scores)])

            # print("Debugggg:", cnt)
            
            mcc_tmp = matthews_corrcoef(y_test, y_pred_adv_tmp)
            ndr_tmp = calculate_accuracy_for_label(y_test, y_pred_adv_tmp, -1)
            
            # if np.mean([mcc_tmp*2,ndr_tmp]) > np.mean([mcc*2,ndr]):
            if np.mean([mcc_tmp*5,ndr_tmp]) > np.mean([mcc*5,ndr]): # For CICIOT2023
                y_pred_adv = y_pred_adv_tmp
                mcc = mcc_tmp
                ndr = ndr_tmp
                thr[id] = x
        y_pred = y_pred_adv
    return y_pred_adv, mcc, thr

    
    # for x in np.linspace(min_th, max_th, num=__step):
    #     y_pred_adv_tmp = np.array([-1 if sc > x else y for y, sc in zip(y_pred,scores)])
    #     mcc_tmp = matthews_corrcoef(y_test, y_pred_adv_tmp)
    #     ndr_tmp = calculate_accuracy_for_label(y_test, y_pred_adv_tmp, -1)
    #     if np.mean([mcc_tmp*2,ndr_tmp]) > np.mean([mcc*2,ndr]):
    #         y_pred_adv = y_pred_adv_tmp
    #         mcc = mcc_tmp
    #         ndr = ndr_tmp
    #         thr = x
    # return y_pred_adv, mcc, thr
    
### Model parameter profile

__Prameter_profile = { 
    'LOF': {
        'n_neighbors': 20,
        'n_jobs':-1,
        'contamination': 0.1
    },
    'IsF': {
        'n_estimators': 100,
        'contamination': 0.1,
        'random_state':42
    },
    'OCSVM': {
        'kernel': "linear",
        'contamination': 0.1
    },
    'RF': {
        'n_estimators': 100,
        'random_state': 42
    },
    'LGBM': {
        'n_estimators': 100,
        'random_state': 42,
        'verbosity':-1
    },
    'MLP': {
        'random_state': 42
    },
    'SGD': {
        'random_state': 42
    },
    'NuSVC': {
        'nu': 0.2, 
        'kernel': "rbf", 
        'random_state': 42
    },
    'SVC': {
        'C': 0.2, 
        'kernel': "rbf", 
        'random_state': 42
    },
    'LDA': {
         'solver': 'svd'
    },
    'KNN': {
         'n_neighbors': 5
    },
    'NC': {
        'metric': 'euclidean'
    },
    'GauNB': {
        'var_smoothing': 1e-09
    },
    'RNC': {
        'outlier_label': 'most_frequent'
    }
}


def Get_base_model(name, param):
    model = None
    if name == "RF":
        model = RandomForestClassifier(**param)
    if name == "LGBM":
        model = LGBMClassifier(**param)
    if name == "MLP":
        model = MLPClassifier(**param)
    if name == "LOF":
        model = LOF(**param)
    if name == "IsF":
        model = IForest(**param)
    if name == "OCSVM":
        model = OCSVM(**param)
    if name == "SGD":
        model = SGDClassifier(**param)
    if name == "NuSVC":
        model = NuSVC(**param)
    if name == "LDA":
        model = LinearDiscriminantAnalysis(**param)
    if name == "KNN":
        model = KNeighborsClassifier(**param)
    if name == "NC":
        model = NearestCentroid(**param)
    if name == "GauNB":
        model = GaussianNB(**param)
    if name == "RNC":
        model = RadiusNeighborsClassifier(**param)
    
    return model

In [229]:
def gaussian_elimination(vectors, threshold=1e-10):
        """Performs Gaussian elimination while preserving the order of vectors.
    
        Args:
            vectors: A numpy array where each row represents a vector.
            threshold: A value below which numbers are considered zero for numerical stability.
    
        Returns:
            Two lists:
                - A list of linearly independent vectors in their original order.
                - A list of indices indicating the positions of the linearly independent vectors
                  in the original input array.
        """
    
        matrix = vectors.copy()
        num_rows, num_cols = matrix.shape
        independent_indices = []  # Track indices of independent vectors
    
        max_rank = min(num_rows, num_cols)
    
        row = 0
        for col in range(max_rank):
            # Find the pivot row (the row with the largest absolute value in the current column)
            pivot_row = row
            for i in range(row + 1, num_rows):
                if abs(matrix[i, col]) > abs(matrix[pivot_row, col]):
                    pivot_row = i
    
            # If the pivot element is close to zero, skip this column (the vector is linearly dependent)
            if abs(matrix[pivot_row, col]) < threshold:
                continue
    
            # Swap the current row with the pivot row ONLY IF THEY ARE DIFFERENT
            if pivot_row != row:
                matrix[[row, pivot_row]] = matrix[[pivot_row, row]]
    
            # Normalize the pivot row (make the pivot element equal to 1)
            pivot = matrix[row, col]
            matrix[row] /= pivot
    
            # Eliminate the elements below the pivot
            for i in range(row + 1, num_rows):
                factor = matrix[i, col]
                matrix[i] -= factor * matrix[row]
    
            independent_indices.append(pivot_row)  # Store the original index
            row += 1
    
        # Extract linearly independent vectors (non-zero rows) in their original order
        independent_vectors = [vectors[i] for i in independent_indices]
    
        return independent_vectors, independent_indices


def form_independent(X_train, y_train, kernel):
    dataX = []
    datay = []
    labels = np.unique(y_train)
    # print("Debug ===== labels:", labels)
    for label in labels:
        mask = np.array([y == label for y in y_train])
        data = X_train[mask]
        if kernel is not None:
            data = metrics.pairwise_kernels(data, metric=kernel)
            data = np.array(data)
        _, indicates = gaussian_elimination(data)
        # print(type(data), data.shape)
        dataX = dataX + X_train[mask][indicates].tolist()
        datay = datay + [label]*len(indicates)
    # print("Debug ===== dataX:", np.array(dataX).shape)
    return np.array(dataX), np.array(datay)

In [230]:
def preprocess_data(drop_cls, data, __poly):
    datasets = data.to_numpy()
    labels = datasets[:,-1]
    dataset = datasets[:,:-1]
        
    logger.info("Distribution of Labels:")
    logger.info(np.unique(labels, return_counts=True))
    # print(type(dataset))
    print("Dataset size:")
    print(dataset.shape)

    ## ========================== Running Main Model ================================================
    if __DATASET == "IoTID20":
        # dataset[np.isinf(dataset)] = np.nan
        dataset[dataset == -np.inf] = np.nan
        dataset[dataset == np.inf] = np.nan
        mean_imputer_X = SimpleImputer(strategy="mean")
        dataset = mean_imputer_X.fit_transform(dataset)

    if __poly > 1:
        from sklearn.preprocessing import PolynomialFeatures
        poly = PolynomialFeatures(__poly,interaction_only=True)
        dataset = poly.fit_transform(dataset)
        print(type(dataset),dataset.shape)
    
    X_train, y_train, X_test, y_test, classes_tmp, encoder = prepare_data(dataset, labels, drop_cls)

    if __N_PCA != -1:
        X_train, pca = apply_pca(X_train)
        X_test = pca.transform(X_test)
    if __N_LDA != -1:
        X_train, lda = apply_lda(X_train, y_train)
        X_test = lda.transform(X_test)


    
    scaler =  Get_Scaler(__SCALER)
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)

    X_train = np.nan_to_num(X_train, nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)

    
    # Remove outliers
    X_train, y_train= remove_outliers_lof(X_train, y_train)

   
    

    return X_train, y_train, X_test, y_test, classes_tmp, encoder




def preprocess_data_iv(drop_cls, data, __poly, kernel):
    datasets = data.to_numpy()
    labels = datasets[:,-1]
    dataset = datasets[:,:-1]
        
    logger.info("Distribution of Labels:")
    logger.info(np.unique(labels, return_counts=True))
    # print(type(dataset))
    print("Dataset size:")
    print(dataset.shape)

    ## ========================== Running Main Model ================================================
    if __DATASET == "IoTID20":
        # dataset[np.isinf(dataset)] = np.nan
        dataset[dataset == -np.inf] = np.nan
        dataset[dataset == np.inf] = np.nan
        mean_imputer_X = SimpleImputer(strategy="mean")
        dataset = mean_imputer_X.fit_transform(dataset)

    if __poly > 1:
        from sklearn.preprocessing import PolynomialFeatures
        poly = PolynomialFeatures(__poly,interaction_only=True)
        dataset = poly.fit_transform(dataset)
        print(type(dataset),dataset.shape)
    
    X_train, y_train, X_test, y_test, classes_tmp, encoder = prepare_data(dataset, labels, drop_cls)

    if __N_PCA != -1:
        X_train, pca = apply_pca(X_train)
        X_test = pca.transform(X_test)
    if __N_LDA != -1:
        X_train, lda = apply_lda(X_train, y_train)
        X_test = lda.transform(X_test)


    
    scaler =  Get_Scaler(__SCALER)
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)

    X_train = np.nan_to_num(X_train, nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)

    
    # Remove outliers
    X_train, y_train = remove_outliers_lof(X_train, y_train)

    if __poly != -1:
        X_train, y_train = form_independent(X_train, y_train, kernel)

    return X_train, y_train, X_test, y_test, classes_tmp, encoder


def Running_Experiment_Novelty_Realistic_baseline(drop_cls, X_train, y_train, X_test, y_test, encoder):
    ## ========================== Create Dataset ================================================
    # print("========================== Beginning Testcase ==========================")
    # res = []
    
    # print("Drop class", drop_cls)
    
    # X_train, y_train, X_test, y_test, classes_tmp, encoder = preprocess_data_iv(drop_cls, data, __POLY)

    __Label_use_name =  list(encoder.classes_)
    print("========================== Running KNFST ==========================")

    # print("Running KNFST")
    # print("Scaler mode:", __SCALER)

    
    t0 = time.time()
    proj, target_points = train_knfst(X_train, y_train, learn)
    
    t1 = time.time()

    mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix, y_pred_adv, y_pred_bina, scores, threshold, test_time = test_knfst_novelty(proj, target_points, X_train, X_test, y_test, encoder)
    t2 = time.time()
    
    print("Training time for KNFST:", t1-t0)
    print("Testing time:", t2-t1)
    
    print("MCC score:", mcc)
    # print("NDR score:", ndr)
    # print("AUC for Novelty score:", auc_n)
    # print("Confusion matrix")
    # print(confusion_matrix)
    # print("Classification report")
    # print(classification_report)

    res.append(["KNFST", mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix, t1-t0, t2-t1])

    # print("*************************************** End Round ***************************************")
    
    return res

def Running_Experiment_Novelty_HHH(drop_cls, X_train, y_train, X_test, y_test, encoder):
    ## ========================== Create Dataset ================================================
    # print("========================== Beginning Testcase ==========================")
    res = []
    
    # print("Drop class", drop_cls)
    
    # X_train, y_train, X_test, y_test, classes_tmp, encoder = preprocess_data(drop_cls, data)
    # X_train, y_train, X_test, y_test, classes_tmp, encoder = preprocess_data_iv(drop_cls, data, __POLY, __KERNEL)
    
    # if __MODE == "Novel":
    #     __Label_use_name =  ["-1"] + list(encoder.classes_)
    # else:
    #     __Label_use_name =  list(encoder.classes_)
    __Label_use_name =  list(encoder.classes_)
    
    print("========================== Running Proposed model ==========================")

    print("Running New Model")
    print("Kernel mode:", __KERNEL)
    print("Scaler mode:", __SCALER)
    
    model = HHHv2()
    
    t0 = time.time()
    
    model.fit(X_train,y_train, kernel = __KERNEL)
    
    t1 = time.time()

    y_preds, y_scores = model.predict(X_test)
    
    
    t2 = time.time()
    mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix = model.evaluating(X_test, y_test, __Label_use_name)

    
    
    print("Training time:", t1 - t0)
    print("Testing time:", t2 - t1)
    
    print("MCC score:", mcc)
    print("NDR score:", ndr)
    print("AUC for Novelty score:", auc_n)
    # print("Confusion matrix")
    # print(confusion_matrix)
    # print("Classification report")
    # print(classification_report)
    

    res.append(["HHHv2", mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix, t1 - t0, t2 - t1])

    # print("*************************************** End Round ***************************************")
    
    return res

def Running_Experiment_FTSPROJECTION_HHH(drop_cls, data):
    ## ========================== Create Dataset ================================================
    print("========================== Beginning Testcase ==========================")
    res = []
    
    print("Drop class", drop_cls)
    
    X_train, y_train, X_test, y_test, classes_tmp, encoder = preprocess_data_iv(drop_cls, data, __POLY)
    # print("DEbug, X_train before:", X_train.shape)
    # print("DEbug, X_test before:", X_test.shape)
    
    if __MODE == "Novel":
        __Label_use_name =  ["-1"] + list(encoder.classes_)
    else:
        __Label_use_name =  list(encoder.classes_)

    print("========================== Running Feature projection model ==========================")

    
    print("Running Projection Model")
    print("Kernel mode:", __KERNEL)
    print("Scaler mode:", __SCALER)
    
    model = HHHv2()
    
    model.fit(X_train.copy(),y_train.copy(), kernel = __KERNEL)

    y_preds, y_scores = model.predict(X_train)

    y_scores = np.min(y_scores, axis=1)


    percentile_90 = np.percentile(y_scores, __KEEP_RATE)
    mask = (y_scores < percentile_90)

    X_train = X_train[mask]
    y_train = y_train[mask]
    
    # plt.plot(y_scores)
    # plt.title('Line Chart of NumPy Array')
    # plt.xlabel('Index')
    # plt.ylabel('Value')
    # plt.show()

    # print(y_scores)
    # return
    X_train = model.transform(X_train.copy())
    X_test = model.transform(X_test.copy())

    # print("DEbug, X_train after:", X_train.shape)
    # print("DEbug, X_test after:", X_test.shape)
    
    for model_name in __MODEL_SUP:
        model = Get_base_model(model_name,__Prameter_profile[model_name])
        
        t0 = time.time()
        
        # model.fit(X_train,y_train, kernel = __KERNEL)
        model, _ = ML_train(X_train, y_train, model)
        
        t1 = time.time()
    
        _, y_pred, _ = ML_test(X_test, y_test, model)
        
        t2 = time.time()
        # mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix = model.evaluating(X_test, y_test, __Label_use_name)
        mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix = Model_evaluating(y_test, y_pred, __Label_use_name)
        ndr = -1
        auc_n = -1
        print("Training time:", t1 - t0)
        print("Testing time:", t2 - t1)
        print("MCC score:", mcc)
        # except:
        #     continue

        res.append([model_name + "_FTSP", mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix, t1 - t0, t2 - t1])
    # print("*************************************** End Round ***************************************")
    
    return res



def Running_Experiment_Novelty_Competitor_Sup(drop_cls, X_train, y_train, X_test, y_test, encoder):
    ## ========================== Create Dataset ================================================
    # print("========================== Beginning Testcase ==========================")
    res = []
    
    # print("Drop class", drop_cls)
    
    # X_train, y_train, X_test, y_test, classes_tmp, encoder = preprocess_data_iv(drop_cls, data, __POLY, __KERNEL)
    
    __Label_use_name =  list(encoder.classes_)

    print("========================== Running Competitor Supervised Model ==========================")

 
    for model_name in __MODEL_SUP:
        print(f"=== Model {model_name} ===")
        model = Get_base_model(model_name,__Prameter_profile[model_name])
        try:
            t0 = time.time()
            
            # model.fit(X_train,y_train, kernel = __KERNEL)
            model, _ = ML_train(X_train, y_train, model)
            
            t1 = time.time()
        
            _, y_pred, _ = ML_test(X_test, y_test, model)
            
            t2 = time.time()
            # mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix = model.evaluating(X_test, y_test, __Label_use_name)
            mcc, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix = Model_evaluating(y_test, y_pred, __Label_use_name)
            ndr = -1
            auc_n = -1
            print("Training time:", t1 - t0)
            print("Testing time:", t2 - t1)
            print("MCC score:", mcc)
        except:
            continue

        res.append([model_name, mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix, t1 - t0, t2 - t1])

    print("*************************************** End Round ***************************************")
    
    return res

In [231]:
## Load Data block

__SEED = __DEFAULT_RANDOM_SEED
seedEverything(__DEFAULT_RANDOM_SEED)

__PREFIX_DIR = "/home/jupyter-hanx"
__WORKING_DIR = f"{__PREFIX_DIR}/HHH"
__DATASET = "IoTID20"
__DATASETS = ["BoT_IoT","ToN_IoT","N_BaIoT","UNSW_NB15",'CIC_IoT2023','IoTID20']
__DATA_DIR = os.path.join(__PREFIX_DIR,'Datasets')
__LIMIT_CNT = 2000
__TARGET = "Label"
__MODE = "Unsup" # "Novel"

logger = setup_logger(level=logging.WARNING)



In [232]:
df = load_ids_by_name(__DATASET)

print(df[__TARGET].unique())

print(df[__TARGET].value_counts())

df.head(5)

['Mirai' 'DoS' '0Normal' 'Scan' 'MITM ARP Spoofing']
Label
Mirai                2000
DoS                  2000
0Normal              2000
Scan                 2000
MITM ARP Spoofing    2000
Name: count, dtype: int64


,Src_Port,Dst_Port,Protocol,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,TotLen_Fwd_Pkts,TotLen_Bwd_Pkts,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,...,Fwd_Seg_Size_Min,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label
0,60424,443,6,130,1,1,261.0,0.0,261.0,261.0,...,0,0.0,0.0,0.0,0.0,130.0,0.000000,130.0,130.0,Mirai
1,52976,9010,6,80,1,1,480.0,0.0,480.0,480.0,...,0,0.0,0.0,0.0,0.0,80.0,0.000000,80.0,80.0,Mirai
2,56204,9020,6,157,0,3,0.0,4164.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,78.5,4.949747,82.0,75.0,Mirai
3,52930,9020,6,150,2,1,0.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,75.0,1.414214,76.0,74.0,Mirai
4,9020,56211,6,150,0,3,0.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,75.0,5.656854,79.0,71.0,Mirai


In [233]:
df.shape

(10000, 80)

In [234]:
__DATA_TYPE = __DATASET
# __classes = np.sort(df[__TARGET].unique())
__KERNEL = "rbf"
__KNN = 20
__POLY = 0
__D = 0
__N_PCA = -1
__N_LDA = -1
__SCALER = "MinMaxScaler"
__KEEP_RATE = 100
# StandardScaler, LabelEncoder, QuantileTransformer, MinMaxScaler, Normalizer

In [235]:
# X_train, y_train, X_test, y_test, classes, encoder = prepare_data(data, label, "*")
# X_train, y_train, X_test, y_test, _, encoder = preprocess_data(["*"], df.copy())
# encoder.classes_

## Running model

In [236]:
__scaler = ['QuantileTransformer', 'StandardScaler', 'MinMaxScaler'] # 'QuantileTransformer', 'StandardScaler', 'MinMaxScaler', 'Normalizer'
__kernel = [None,'rbf','poly','linear','sigmoid'] #'rbf','poly','linear','sigmoid',None
# __MODEL_SUP = ["MLP","LGBM","NuSVC","SGD","RF"]
__MODEL_SUP = ["KNN","LDA","NuSVC","SGD","GauNB", "NC", "RNC"] #"KNN","LDA","MLP","LGBM","RF","NuSVC","SGD","GauNB", "NC", "RNC"
# __range = np.concatenate((["*"], __classes), axis=0)
__range = ["*"]
__res = []
__poly = []


if df.shape[1] < 30:
    __poly = [3, 2, 0, -1]
else:
    __poly = [2, 0, -1]

for scaler in __scaler:
    __SCALER = scaler
    
    for poly in __poly:
        # print(f"================================================= Scaler {scaler} - Poly {__poly} =================================================")
        __POLY = poly

        for kernel in __kernel:
            __KERNEL = kernel

            
            print(f"================================================= Scaler {__SCALER} - Poly {__POLY} - Kernel {__KERNEL} =================================================")
            X_train, y_train, X_test, y_test, classes_tmp, encoder = preprocess_data_iv("*",df.copy(), __POLY, __KERNEL)
            print("Train shape:", X_train.shape)
            print("Test shape:", X_test.shape)
            
            # if __poly == 3 and __KERNEL is not None:
            #     continue
           
            if __KERNEL == 'rbf' and __poly < 2:
                res_tmp = Running_Experiment_Novelty_Realistic_baseline("*",X_train.copy(), y_train.copy(), X_test.copy(), y_test.copy(), encoder)
                for x in res_tmp:
                    __res.append([__DATA_TYPE, __POLY, __KERNEL, __SCALER] + x)
                    
            res_tmp = Running_Experiment_Novelty_Competitor_Sup("*",X_train.copy(), y_train.copy(), X_test.copy(), y_test.copy(), encoder)
            for x in res_tmp:
                __res.append([__DATA_TYPE, __POLY, __KERNEL, __SCALER] + x)
             
            # res_tmp = Running_Experiment_Novelty_HHH("*",X_train.copy(), y_train.copy(), X_test.copy(), y_test.copy(), encoder)
            # __KERNEL = "None" if (__KERNEL is None) else __KERNEL
            # for x in res_tmp:
            #     __res.append([__DATA_TYPE, __POLY, __KERNEL, __SCALER] + x)

            # res_tmp = Running_Experiment_FTSPROJECTION_HHH(drop_clss, df.copy())
            # __KERNEL = "None" if (__KERNEL is None) else __KERNEL
            # for x in res_tmp:
            #     __res.append([__DATA_TYPE, drop_clss, __KERNEL, __SCALER] + x)
            print ("================================================ End Round =================================================")


    

res_df = pd.DataFrame(__res, columns = ["Data Type", "Poly", "Kernel", "SCALER",
                                     "Model", "MCC", "NDR", "AUC_N", "ACC", "TPR Macro",
                                    "FPR", "PPV Macro", "F1 Macro", "AUC", "CLS Report",
                                    "CFS Matrix","Training time", "Test time"])
# res_df.to_csv(f"./HHH/Pre_Result/HHHv2_Projection_{__KEEP_RATE}Remain_{__DATA_TYPE}_{__LIMIT_CNT}_PCA{__N_PCA}_LDA{__N_LDA}_UsingSpareMatrix.csv",index = False)

res_df.to_csv(f"./tmp/{__DATASET}_{__LIMIT_CNT}_extend_Competitors.csv")

================================================= Scaler QuantileTransformer - Poly 2 - Kernel None =================================================
Dataset size:
(10000, 79)
<class 'numpy.ndarray'> (10000, 3161)
['0Normal' 'DoS' 'MITM ARP Spoofing' 'Mirai' 'Scan']
Train shape: (3571, 3161)
Test shape: (2000, 3161)
========================== Running Competitor Supervised Model ==========================
=== Model KNN ===
Training time: 0.004706859588623047
Testing time: 0.19778871536254883
MCC score: 0.7092752798632731
=== Model LDA ===
Training time: 6.74076247215271
Testing time: 0.004941463470458984
MCC score: 0.761410848413469
=== Model MLP ===
Training time: 7.3377580642700195
Testing time: 0.034635305404663086
MCC score: 0.7402487791754337
=== Model LGBM ===
Training time: 13.9643075466156
Testing time: 0.03906106948852539
MCC score: 0.9306491390251005
=== Model RF ===
Training time: 2.188206911087036
Testing time: 0.02799201011657715
MCC score: 0.8713842579801019
=== Model NuSV

/tmp/ipykernel_54155/2950139054.py:49: RuntimeWarning: invalid value encountered in divide
  FPR = FP/(FP+TN) *100.


Train shape: (269, 79)
Test shape: (2000, 79)
========================== Running Competitor Supervised Model ==========================
=== Model KNN ===
Training time: 0.0006861686706542969
Testing time: 0.07590723037719727
MCC score: 0.46763505832813307
=== Model LDA ===
Training time: 0.002663850784301758
Testing time: 0.00027871131896972656
MCC score: 0.5766073798245783
=== Model MLP ===


/opt/tljh/user/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Training time: 0.1384565830230713
Testing time: 0.0008397102355957031
MCC score: 0.6528784555933507
=== Model LGBM ===
Training time: 0.4027259349822998
Testing time: 0.011815786361694336
MCC score: 0.7475223879557287
=== Model RF ===
Training time: 0.08395910263061523
Testing time: 0.012238502502441406
MCC score: 0.6514882312612386
=== Model NuSVC ===
Training time: 0.006726980209350586
Testing time: 0.017069101333618164
MCC score: 0.5228264919644074
=== Model SGD ===
Training time: 0.0039823055267333984
Testing time: 0.0004990100860595703
MCC score: 0.5505784843698706
*************************************** End Round ***************************************
========================== Running Proposed model ==========================
Running New Model
Kernel mode: None
Scaler mode: QuantileTransformer
Training time: 0.00982356071472168
Testing time: 0.006246328353881836
MCC score: 0.5865872840930739
NDR score: -1
AUC for Novelty score: -1
===============================================

...

Training time: 3.252748489379883
Testing time: 0.0024840831756591797
MCC score: 0.7870953516263592
=== Model LGBM ===
Training time: 1.4076049327850342
Testing time: 0.008055686950683594
MCC score: 0.9506722377004014
=== Model RF ===
Training time: 1.0311577320098877
Testing time: 0.017649173736572266
MCC score: 0.8704369602266496
=== Model NuSVC ===
Training time: 1.3171072006225586
Testing time: 0.27840566635131836
MCC score: 0.7758648439414517
=== Model SGD ===
Training time: 0.10225129127502441
Testing time: 0.00057220458984375
MCC score: 0.663513620367942
*************************************** End Round ***************************************
========================== Running Proposed model ==========================
Running New Model
Kernel mode: poly
Scaler mode: MinMaxScaler
Training time: 1171.0766501426697
Testing time: 0.1703343391418457
MCC score: 0.793303986291376
NDR score: -1
AUC for Novelty score: -1
================================================ End Round =========

/opt/tljh/user/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Training time: 2.9562737941741943
Testing time: 0.0008082389831542969
MCC score: 0.7870953516263592
=== Model LGBM ===
Training time: 1.4025084972381592
Testing time: 0.008516073226928711
MCC score: 0.9506722377004014
=== Model RF ===
Training time: 0.6693084239959717
Testing time: 0.016521453857421875
MCC score: 0.8704369602266496
=== Model NuSVC ===
Training time: 1.2029345035552979
Testing time: 0.27858924865722656
MCC score: 0.7758648439414517
=== Model SGD ===
Training time: 0.09160494804382324
Testing time: 0.0005793571472167969
MCC score: 0.663513620367942
*************************************** End Round ***************************************
========================== Running Proposed model ==========================
Running New Model
Kernel mode: linear
Scaler mode: MinMaxScaler
Training time: 284.1332323551178
Testing time: 0.029722929000854492
MCC score: 0.6587307723303638
NDR score: -1
AUC for Novelty score: -1
================================================ End Round ==

/opt/tljh/user/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Training time: 2.9914376735687256
Testing time: 0.0008294582366943359
MCC score: 0.7870953516263592
=== Model LGBM ===
Training time: 1.610588550567627
Testing time: 0.008811235427856445
MCC score: 0.9506722377004014
=== Model RF ===
Training time: 0.7767598628997803
Testing time: 0.022016286849975586
MCC score: 0.8704369602266496
=== Model NuSVC ===
Training time: 1.3750221729278564
Testing time: 0.2658233642578125
MCC score: 0.7758648439414517
=== Model SGD ===
Training time: 0.091827392578125
Testing time: 0.0005316734313964844
MCC score: 0.663513620367942
*************************************** End Round ***************************************
========================== Running Proposed model ==========================
Running New Model
Kernel mode: sigmoid
Scaler mode: MinMaxScaler
Training time: 266.8388867378235
Testing time: 0.11303472518920898
MCC score: 0.4774380609714708
NDR score: -1
AUC for Novelty score: -1
================================================ End Round ======

## Ignore

In [237]:
# import sympy as sp
# import numpy as np

# def gaussian_elimination_preserve_order(vectors, threshold=1e-10):
#     """Performs Gaussian elimination while preserving the order of vectors.

#     Args:
#         vectors: A numpy array where each row represents a vector.
#         threshold: A value below which numbers are considered zero for numerical stability.

#     Returns:
#         Two lists:
#             - A list of linearly independent vectors in their original order.
#             - A list of indices indicating the positions of the linearly independent vectors
#               in the original input array.
#     """

#     matrix = vectors.copy()
#     num_rows, num_cols = matrix.shape
#     independent_indices = []  # Track indices of independent vectors

#     row = 0
#     for col in range(num_cols):
#         # Find the pivot row (the row with the largest absolute value in the current column)
#         pivot_row = row
#         for i in range(row + 1, num_rows):
#             if abs(matrix[i, col]) > abs(matrix[pivot_row, col]):
#                 pivot_row = i

#         # If the pivot element is close to zero, skip this column (the vector is linearly dependent)
#         if abs(matrix[pivot_row, col]) < threshold:
#             continue

#         # Swap the current row with the pivot row ONLY IF THEY ARE DIFFERENT
#         if pivot_row != row:
#             matrix[[row, pivot_row]] = matrix[[pivot_row, row]]

#         # Normalize the pivot row (make the pivot element equal to 1)
#         pivot = matrix[row, col]
#         matrix[row] /= pivot

#         # Eliminate the elements below the pivot
#         for i in range(row + 1, num_rows):
#             factor = matrix[i, col]
#             matrix[i] -= factor * matrix[row]

#         independent_indices.append(pivot_row)  # Store the original index
#         row += 1

#     # Extract linearly independent vectors (non-zero rows) in their original order
#     independent_vectors = [vectors[i] for i in independent_indices]

#     return independent_vectors, np.array(independent_indices)




    






# # # Example usage
# # vectors = np.array([[1., 0., 0.],
# #                     [0., 1., 0.],
# #                     [1., 1., 0.],
# #                     [2., 0., 0.]])
# # independent_vectors, indices = gaussian_elimination_preserve_order(vectors)

# # print("Linearly independent vectors:")
# # for v in independent_vectors:
# #     print(v)

# # X_train, X_test, y_train, y_test = generate_independently_vectors(10, 2500, 2000, 2000)
# __SEED = 42
# __DATA_TYPE = "MIX CLUSTER"


# X_train, label = generate_multiclass_dataset(10, 5, 1000, 1000, 0)
# print(X_train.shape)


# t0 = time.time()
# independent_vectors, independent_indices = gaussian_elimination_preserve_order(X_train)
# print(independent_indices.shape)
# t1 = time.time()
# print("time:", t1-t0)


# m = sp.Matrix(X_train)
# t0 = time.time()
# m_rref, pivots = m.rref() # Compute reduced row echelon form (rref).
# t1 = time.time()
# print(pivots.shape)
# print("time:", t1-t0)

# for x in pivots:
#     if x not in independent_indices:
#         print("WRONG")
#         break

In [238]:
# X_train, y_train, X_test, y_test, _, encoder = preprocess_data(["*"], df.copy())

In [239]:
# import numpy as np


# def conjugate_grad_1(A, b, x=None):
#     n = len(b)
#     if x is None:
#         x = np.ones(n)
    
#     r = np.dot(A, x) - b
#     p = - r
#     r_k_norm = np.dot(r, r)
    
#     for i in trange(3*n):
        
#         Ap = np.dot(A, p)
        
#         alpha = r_k_norm / np.dot(p, Ap)
        
#         x += alpha * p
#         r += alpha * Ap
        
#         r_kplus1_norm = np.dot(r, r)
        
#         beta = r_kplus1_norm / r_k_norm
        
#         r_k_norm = r_kplus1_norm

#         print(f"Current iter {i} , loss: {r_kplus1_norm}", end='\r')
#         if r_kplus1_norm < 1e-12:
#             print( 'Itr:', i)
#             break
        
#         p = beta * p - r
    
#     return x


# def conjugate_grad_2(A, b, x=None):
#     n = len(b)
#     if x is None:
#         x = np.ones(n)
    
#     r = A.dot(x) - b
#     p = - r
#     r_k_norm = r.dot(r)
    
#     for i in trange(3*n):
        
#         Ap = A.dot(p)
        
#         alpha = r_k_norm / p.dot(Ap)
        
#         x += alpha * p
#         r += alpha * Ap
        
#         r_kplus1_norm = r.dot(r)
        
#         beta = r_kplus1_norm / r_k_norm
        
#         r_k_norm = r_kplus1_norm

#         print(f"Current iter {i} , loss: {r_kplus1_norm}", end='\r')
#         if r_kplus1_norm < 1e-12:
#             print( 'Itr:', i)
#             break
        
#         p = beta * p - r
    
#     return x






In [240]:
# n = X_train.shape[0]
        
# b = y_train.T

# A = metrics.pairwise_kernels(X_train, metric="rbf") + np.eye(n,n) * 1e-5

In [241]:
# t0 = time.time()
# x1 = conjugate_grad_1(A, b)
# print("Running time:", time.time() - t0)


In [242]:
# t0 = time.time()
# x2 = conjugate_grad_2(A, b)
# print("Running time:", time.time() - t0)


In [243]:
# # print(x1)
# print(x2)
# print(x3)
# print(np.sum(x2 - x3))

In [244]:
# print("========================== Running baseline ==========================")
# encoder = LabelEncoder()
# y_train = encoder.fit_transform(y_train)

# print("Running KNFST")

# t0 = time.time()
# proj, target_points = train_knfst(X_train_scaled, y_train, learn)

# t1 = time.time() - t0
# t0 = time.time()
# mcc, ndr, auc_n, acc, tpr_func, fpr, ppv_func, f1_func, auc_func, clf_report, cnf_matrix, y_pred_adv, y_pred_bina, scores, threshold, test_time = test_knfst_novelty(proj, target_points, X_train_scaled, X_test_scaled, y_test, encoder)
# t2 = time.time() - t0

# print("Training time for KNFST:", t1)
# print("Testing time:", t2)

# print("MCC score:", mcc)
# print("NDR score:", ndr)
# print("AUC for Novelty score:", auc_n)


In [245]:
# import numpy as np
# import tensorflow as tf
# from tensorflow.keras import layers, models

# # Build CNN model
# model = models.Sequential([
#     layers.Conv2D(32, (3, 3), activation='relu', input_shape=(__fts_shape, __fts_shape, 3)),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(64, (3, 3), activation='relu'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(128, (3, 3), activation='relu'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(128, (3, 3), activation='relu'),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D((2, 2)),
#     layers.Flatten(),
#     layers.Dense(512, activation='relu'),
#     layers.Dense(256, activation='relu'),
#     layers.Dense(128, activation='relu'),
#     layers.Dense(categories_num, activation='softmax')
# ])

# # Compile model
# # model.compile(optimizer='adam',
# #               loss='sparse_categorical_crossentropy',
# #               metrics=['sparse_categorical_accuracy'])
# model.compile(optimizer='adam',
#              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
#              metrics=['sparse_categorical_accuracy'])

# model.fit(X_train_dl, y_train, epochs=10, batch_size=32, validation_data=(X_test_dl, y_test))

# # Evaluate model
# test_loss, test_acc = model.evaluate(X_test_dl)
# print(f'Test accuracy: {test_acc}')


# # Evaluate model
# y_pred_dl = model.predict(X_test_dl)
# y_pred_dl = np.argmax(y_pred_dl, axis=1)
# y_test_dl_cv = np.argmax(y_test_dl, axis=1)
# mcc = matthews_corrcoef(y_test, y_pred_dl)
# print(f'Matthews correlation coefficient: {mcc}')
